In [1]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime
import pandas as pd

## 1. Configuration et connexion

In [2]:
# Identifiants
USERNAME = "0838827"
PASSWORD = "CheeGliFFSU2!"

# URLs de base
BASE_URL = "https://gestion.mysportu.com"
LOGIN_URL = f"{BASE_URL}/login"

# Créer une session pour maintenir les cookies
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

In [7]:
# Test des différentes URLs de base MySportU
test_urls = [
    "https://gestion.mysportu.com",
    "https://mysportu.com",
    "https://www.mysportu.com",
    "https://sport-u.com",
    "https://sport-u.com/mysportu",
]

for url in test_urls:
    try:
        resp = requests.get(url, timeout=10, allow_redirects=True, headers={
            'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'
        })
        print(f"{url}: {resp.status_code} -> {resp.url}")
    except Exception as e:
        print(f"{url}: {type(e).__name__}")

https://gestion.mysportu.com: 200 -> https://gestion.mysportu.com/auth/login
https://mysportu.com: 200 -> https://sport-u.com/mysportu/
https://www.mysportu.com: 200 -> https://sport-u.com/mysportu/
https://sport-u.com: 200 -> https://sport-u.com/
https://sport-u.com/mysportu: 200 -> https://sport-u.com/mysportu/


In [8]:
# Accéder à la vraie page de login
BASE_URL = "https://gestion.mysportu.com"
LOGIN_URL = f"{BASE_URL}/auth/login"

response = session.get(LOGIN_URL)
print(f"Status: {response.status_code}")
print(f"URL: {response.url}")

# Parser pour trouver le formulaire de login
soup = BeautifulSoup(response.text, 'html.parser')

# Chercher le token CSRF
csrf_token = None
csrf_input = soup.find('input', {'name': '_token'})
if csrf_input:
    csrf_token = csrf_input.get('value')
    print(f"Token CSRF trouvé: {csrf_token[:30]}...")
else:
    print("Token CSRF non trouvé via input _token")
    # Chercher d'autres patterns
    meta_csrf = soup.find('meta', {'name': 'csrf-token'})
    if meta_csrf:
        csrf_token = meta_csrf.get('content')
        print(f"Token CSRF trouvé via meta: {csrf_token[:30]}...")

# Afficher tous les inputs du formulaire
form = soup.find('form')
if form:
    print("\nInputs du formulaire:")
    for inp in form.find_all('input'):
        print(f"  {inp.get('name')} = {inp.get('value', '')[:30] if inp.get('value') else ''} (type: {inp.get('type')})")

Status: 200
URL: https://gestion.mysportu.com/auth/login
Token CSRF trouvé: wzZkqH83KFkiLqd8nzA9ATxj75QCmu...

Inputs du formulaire:
  _token = wzZkqH83KFkiLqd8nzA9ATxj75QCmu (type: hidden)
  username =  (type: text)
  password =  (type: password)


In [9]:
# Tentative de connexion
login_data = {
    '_token': csrf_token,
    'username': USERNAME,
    'password': PASSWORD,
}

response = session.post(LOGIN_URL, data=login_data, allow_redirects=True)
print(f"Status: {response.status_code}")
print(f"URL après login: {response.url}")
print(f"Cookies: {list(session.cookies.keys())}")

# Vérifier si on est connecté
if 'login' in response.url.lower() or 'auth' in response.url.lower():
    print("\n❌ Connexion échouée - toujours sur la page de login")
    # Chercher les messages d'erreur
    soup = BeautifulSoup(response.text, 'html.parser')
    errors = soup.find_all(class_=lambda x: x and 'error' in str(x).lower() or 'alert' in str(x).lower())
    for error in errors:
        print(f"Erreur: {error.get_text(strip=True)[:200]}")
else:
    print("\n✅ Connexion réussie!")
    soup = BeautifulSoup(response.text, 'html.parser')
    title = soup.find('title')
    print(f"Titre: {title.text if title else 'N/A'}")

Status: 200
URL après login: https://gestion.mysportu.com
Cookies: ['XSRF-TOKEN', 'ffsu_session']

✅ Connexion réussie!
Titre: FFSU - Accueil


In [10]:
# Explorer les liens disponibles sur la page d'accueil
soup = BeautifulSoup(response.text, 'html.parser')

# Trouver tous les liens
links = soup.find_all('a', href=True)
unique_links = set()
for link in links:
    href = link['href']
    text = link.get_text(strip=True)[:50]
    if href.startswith('/') or 'mysportu' in href or 'gestion' in href:
        unique_links.add((href, text))

print("Liens trouvés sur la page d'accueil:")
for href, text in sorted(unique_links):
    print(f"  {href} - {text}")

Liens trouvés sur la page d'accueil:
  /images/CharteFFSU-CGU.pdf - CGU / RGPD
  /images/CharteFFSU-CGU.pdf - être en accord avec notre charte de protection et 
  https://gestion.mysportu.com - 
  https://gestion.mysportu.com - Accueil
  https://gestion.mysportu.com/arbitrage/designation - Désignations
  https://gestion.mysportu.com/articles - Voir tous nos articles
  https://gestion.mysportu.com/articles/10 - 
  https://gestion.mysportu.com/articles/10 - Lire la suite
  https://gestion.mysportu.com/articles/10 - NOUVEAU REGLEMENT DES CHALLENGES NATIONAUX DES AS
  https://gestion.mysportu.com/articles/3 - 
  https://gestion.mysportu.com/articles/3 - Découvrez my sport U !
  https://gestion.mysportu.com/articles/3 - Lire la suite
  https://gestion.mysportu.com/articles/5 - 
  https://gestion.mysportu.com/articles/5 - Lire la suite
  https://gestion.mysportu.com/articles/5 - PASS SPORT ! 70€ de remise immédiate sur l'inscrip
  https://gestion.mysportu.com/articles/6 - 
  https://gestion.

In [11]:
# Filtrer les liens pertinents (rencontres, matchs, compétitions)
keywords = ['rencontre', 'match', 'competition', 'calendrier', 'journee', 'poule', 'equipe', 'feuille', 'volley']

relevant_links = []
for href, text in unique_links:
    combined = (href + text).lower()
    if any(kw in combined for kw in keywords):
        relevant_links.append((href, text))

print("Liens pertinents trouvés:")
for href, text in sorted(relevant_links):
    print(f"  {href} - {text}")

Liens pertinents trouvés:
  https://gestion.mysportu.com/extractions/arbitrage/designations-par-competition-lieu-personne - Désignations par lieux
  https://gestion.mysportu.com/feuille-de-match - Rencontres
  https://gestion.mysportu.com/manifestations/calendrier-federal - Calendrier fédéral
  https://gestion.mysportu.com/sportif/competitions - Liste des compétitions
  https://gestion.mysportu.com/sportif/equipes/extraction/correspondant - Liste des correspondants
  https://gestion.mysportu.com/sportif/validations/competitions - Validations des compétitions


In [12]:
# Explorer la page des feuilles de match
feuille_url = f"{BASE_URL}/feuille-de-match"
response = session.get(feuille_url)
print(f"Status: {response.status_code}")
print(f"URL: {response.url}")

soup = BeautifulSoup(response.text, 'html.parser')
title = soup.find('title')
print(f"Titre: {title.text if title else 'N/A'}")

# Afficher le contenu principal
print("\n--- Début du contenu ---")
print(response.text[:3000])

Status: 200
URL: https://gestion.mysportu.com/feuille-de-match
Titre: FFSU - Feuille de match

--- Début du contenu ---
<!DOCTYPE html>
<html lang="fr">

<head data-snd="PE6vW5t6DYUX4lb2HnTX69QKaliaVXVB">
    <meta http-equiv="content-type" content="text/html;charset=UTF-8"/>
    <meta charset="utf-8"/>
    <title>FFSU - Feuille de match</title>
	<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">
    <meta content="" name="description"/>
    <meta content="" name="author"/>
    <meta name="csrf-token" content="oWi7ctQbDRpcQV6lin18x94IMA8T2DGZC88HX3Gt">
    <link rel="icon" type="image/x-icon" href="/favicon.ico">
        <link href="https://fonts.googleapis.com/css?family=Roboto:400,300,100,500,700,900" rel="stylesheet" type="text/css">

<link  href="/css/app.css?id=0f1e10e944c4f6d1b54d" rel="stylesheet" type="text/css"/>


<script type="text/javascript" src="/js/vendor.js?id=461314863c1c08b8f668"></script>
<script type="text/javascript" src="/js/app

In [13]:
# Chercher les formulaires de filtrage sur la page
forms = soup.find_all('form')
print(f"Nombre de formulaires: {len(forms)}")

for i, form in enumerate(forms):
    print(f"\n=== Formulaire {i+1} ===")
    print(f"Action: {form.get('action')}")
    print(f"Méthode: {form.get('method')}")
    inputs = form.find_all(['input', 'select'])
    for inp in inputs:
        print(f"  {inp.name}: {inp.get('name')} = {inp.get('value', '')[:30] if inp.get('value') else ''}")

Nombre de formulaires: 3

=== Formulaire 1 ===
Action: https://gestion.mysportu.com/personnes/recherche
Méthode: GET
  input: personnes_q = 

=== Formulaire 2 ===
Action: https://gestion.mysportu.com/licences/recherche
Méthode: GET
  input: licencies_q = 

=== Formulaire 3 ===
Action: https://gestion.mysportu.com/structures/recherche
Méthode: GET
  input: structures_q = 


In [14]:
# Chercher les éléments select (dropdowns) pour filtrer les rencontres
selects = soup.find_all('select')
print(f"Nombre de selects: {len(selects)}")

for sel in selects:
    print(f"\n=== Select: {sel.get('name')} (id: {sel.get('id')}) ===")
    options = sel.find_all('option')[:10]  # Limiter à 10 options
    for opt in options:
        print(f"  {opt.get('value')} - {opt.get_text(strip=True)[:50]}")

Nombre de selects: 0


In [15]:
# Analyser le contenu de la page - chercher des data attributes ou scripts avec données
import re

# Chercher les scripts inline avec des données
scripts = soup.find_all('script')
print(f"Nombre de scripts: {len(scripts)}")

for i, script in enumerate(scripts):
    if script.string:
        content = script.string
        # Chercher des URLs API ou des objets JSON
        if 'api' in content.lower() or 'fetch' in content.lower() or 'axios' in content.lower() or 'rencontre' in content.lower() or 'match' in content.lower():
            print(f"\n=== Script {i+1} (pertinent) ===")
            print(content[:1500] if len(content) > 1500 else content)

Nombre de scripts: 11


In [16]:
# Afficher tous les scripts pour analyse
for i, script in enumerate(scripts):
    if script.string:
        print(f"\n=== Script {i+1} ===")
        print(script.string[:800] if len(script.string) > 800 else script.string)
        print("..." if len(script.string) > 800 else "")


=== Script 3 ===

        window.currentStructureAuth = {"id":2,"nom":"LRSU Auvergne-Rh\u00f4ne-Alpes","type":"ZON","code":"LRAUR","color":"dark","TypeColorLabel":"dark","styled":"<span class=\"text-dark\">\n                \n        <span class=\"badge badge-dark position-relative mr-1\"> LRAUR<\/span>\n        LRSU Auvergne-Rh\u00f4ne-Alpes \n<\/span>\n","type_libelle":"Ligue R\u00e9gionale"}
    


=== Script 4 ===

    const flashbag = new Vue({
        el: '#flashbag_container',
    });

    var messages_types = {};





    flashbag.$refs.flashbag.nouveaux_messages(messages_types);



=== Script 5 ===

        window.avecSocket = 0;
        window.avecSocketPort = 6001;
    


=== Script 7 ===

        $(function () {
                                        const personne_id = '838828';

                $('#Accepte').click(function () {
                    $.get('https://gestion.mysportu.com/accepter-conditions', {personne_id: personne_id}, function (json) {
                    

In [17]:
# Chercher des liens vers des rencontres spécifiques dans le HTML
links_rencontres = soup.find_all('a', href=re.compile(r'/rencontre|/feuille-de-match'))
print(f"Liens vers rencontres trouvés: {len(links_rencontres)}")
for link in links_rencontres[:20]:
    print(f"  {link.get('href')} - {link.get_text(strip=True)[:60]}")

Liens vers rencontres trouvés: 2
  https://gestion.mysportu.com/feuille-de-match - Rencontres
  https://gestion.mysportu.com/feuille-de-match - Rencontres


In [18]:
# Chercher tous les tableaux et divs qui pourraient contenir des données
tables = soup.find_all('table')
print(f"Tableaux trouvés: {len(tables)}")

# Chercher les divs avec des classes qui suggèrent des données
data_divs = soup.find_all(class_=re.compile(r'table|list|grid|data|card|row|rencontre|match', re.I))
print(f"Divs de données trouvés: {len(data_divs)}")

# Afficher les classes les plus courantes
from collections import Counter
all_classes = []
for el in soup.find_all(class_=True):
    all_classes.extend(el.get('class'))
    
class_counts = Counter(all_classes)
print("\nClasses les plus courantes:")
for cls, count in class_counts.most_common(30):
    print(f"  {cls}: {count}")

Tableaux trouvés: 0
Divs de données trouvés: 19

Classes les plus courantes:
  nav-item: 110
  nav-link: 108
  nav: 24
  nav-item-submenu: 23
  nav-group-sub: 23
  mr-1: 18
  icon-stack-text: 17
  icon-stack-check: 14
  ml-3: 13
  text-black-50: 12
  media: 10
  media-body: 10
  font-size-sm: 9
  btn: 9
  d-none: 8
  icon-search4: 8
  badge: 8
  position-relative: 8
  text-center: 8
  d-block: 7
  text-muted: 7
  icon-list: 7
  text-primary: 6
  icon-city: 6
  no-wrap: 6
  d-lg-inline-block: 6
  nav-item-header: 5
  text-uppercase: 5
  font-size-xs: 5
  line-height-xs: 5


In [19]:
# Explorer la page des compétitions
competitions_url = f"{BASE_URL}/sportif/competitions"
response = session.get(competitions_url)
print(f"Status: {response.status_code}")
print(f"URL: {response.url}")

soup = BeautifulSoup(response.text, 'html.parser')
title = soup.find('title')
print(f"Titre: {title.text if title else 'N/A'}")

# Chercher les liens vers des compétitions spécifiques
comp_links = soup.find_all('a', href=re.compile(r'/competition|/sportif'))
print(f"\nLiens compétitions trouvés: {len(comp_links)}")
for link in comp_links[:30]:
    text = link.get_text(strip=True)
    if text and len(text) > 3:
        print(f"  {link.get('href')} - {text[:60]}")

Status: 200
URL: https://gestion.mysportu.com/sportif/competitions
Titre: FFSU - Compétitions

Liens compétitions trouvés: 10
  https://gestion.mysportu.com/sportif/competitions - Liste des compétitions
  https://gestion.mysportu.com/sportif/validations/competitions - Validations des compétitions
  https://gestion.mysportu.com/sportif/equipes/extraction/correspondant - Liste des correspondants
  https://gestion.mysportu.com/sportif/engagement/extraction - Liste des engagements
  https://gestion.mysportu.com/sportif/engagement/validation/saison/2026 - Validation des engagements
  https://gestion.mysportu.com/extractions/sportif/joueurs - Participation des joueurs
  https://gestion.mysportu.com/extractions/sportif/officiels - Participation des officiels
  https://gestion.mysportu.com/extractions/sportif/staffs - Participation du staff
  https://gestion.mysportu.com/sportif/extractions/rosters - Rosters/Listes
  https://gestion.mysportu.com/sportif/sanctions - Sanctions financières


In [20]:
# Afficher le HTML de la page compétitions pour trouver les filtres
print(response.text[:5000])

<!DOCTYPE html>
<html lang="fr">

<head data-snd="PE6vW5t6DYUX4lb2HnTX69QKaliaVXVB">
    <meta http-equiv="content-type" content="text/html;charset=UTF-8"/>
    <meta charset="utf-8"/>
    <title>FFSU - Compétitions</title>
	<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">
    <meta content="" name="description"/>
    <meta content="" name="author"/>
    <meta name="csrf-token" content="oWi7ctQbDRpcQV6lin18x94IMA8T2DGZC88HX3Gt">
    <link rel="icon" type="image/x-icon" href="/favicon.ico">
        <link href="https://fonts.googleapis.com/css?family=Roboto:400,300,100,500,700,900" rel="stylesheet" type="text/css">

<link  href="/css/app.css?id=0f1e10e944c4f6d1b54d" rel="stylesheet" type="text/css"/>


<script type="text/javascript" src="/js/vendor.js?id=461314863c1c08b8f668"></script>
<script type="text/javascript" src="/js/app.js?id=d41d8cd98f00b204e980"></script>

    
    <script async>
        window.currentStructureAuth = {"id":2,"nom":"LRSU Au

In [21]:
# Chercher les formulaires de filtrage et les selects
soup = BeautifulSoup(response.text, 'html.parser')
forms = soup.find_all('form')
print(f"Formulaires: {len(forms)}")

selects = soup.find_all('select')
print(f"Selects: {len(selects)}")

for sel in selects:
    print(f"\n=== Select: {sel.get('name')} (id: {sel.get('id')}) ===")
    options = sel.find_all('option')
    print(f"Options ({len(options)}):")
    for opt in options[:15]:
        print(f"  {opt.get('value')} - {opt.get_text(strip=True)[:60]}")

Formulaires: 3
Selects: 0


In [22]:
# Chercher les tableaux de données ou les listes de compétitions
tables = soup.find_all('table')
print(f"Tableaux: {len(tables)}")

# Chercher des éléments qui pourraient contenir des données de compétitions
divs_content = soup.find_all('div', class_=re.compile(r'content|card|list|container', re.I))
print(f"Divs de contenu: {len(divs_content)}")

# Afficher le contenu principal de la page
main_content = soup.find('main') or soup.find(id='main') or soup.find(class_='content')
if main_content:
    print("\n=== Contenu principal ===")
    print(main_content.get_text(strip=True)[:2000])
else:
    # Chercher le body
    body = soup.find('body')
    if body:
        print("\n=== Texte de la page ===")
        text = body.get_text(separator='\n', strip=True)
        print(text[:3000])

Tableaux: 0
Divs de contenu: 17

=== Contenu principal ===



In [23]:
# Chercher des endpoints API dans le HTML
api_patterns = [
    r'["\']\/api\/[^"\']+["\']',
    r'fetch\(["\'][^"\']+["\']',
    r'axios\.[a-z]+\(["\'][^"\']+["\']',
    r'url:\s*["\'][^"\']+["\']',
    r'href=["\']\/[^"\']*rencontre[^"\']*["\']',
    r'href=["\']\/[^"\']*competition[^"\']*["\']',
]

html_text = response.text
found = set()
for pattern in api_patterns:
    matches = re.findall(pattern, html_text, re.I)
    found.update(matches)

print("Endpoints/URLs trouvés dans le HTML:")
for f in sorted(found):
    print(f"  {f}")

Endpoints/URLs trouvés dans le HTML:


In [24]:
# Chercher si c'est une app Vue.js ou React
frameworks = {
    'vue': r'vue|v-bind|v-model|v-for|:class|@click',
    'react': r'react|jsx|__NEXT|_next',
    'angular': r'ng-|angular',
    'livewire': r'livewire|wire:',
    'alpine': r'x-data|x-bind|x-on|alpine',
}

html_lower = response.text.lower()
print("Frameworks détectés:")
for name, pattern in frameworks.items():
    if re.search(pattern, html_lower):
        print(f"  ✅ {name}")

# Chercher des data-* attributes qui pourraient contenir des URLs
data_attrs = re.findall(r'data-[a-z-]+="[^"]+"', response.text)
print(f"\nData attributes trouvés: {len(data_attrs)}")
for attr in data_attrs[:30]:
    print(f"  {attr}")

Frameworks détectés:

Data attributes trouvés: 32
  data-snd="PE6vW5t6DYUX4lb2HnTX69QKaliaVXVB"
  data-toggle="collapse"
  data-target="#navbar-mobile"
  data-toggle="dropdown"
  data-toggle="dropdown"
  data-nav-type="accordion"
  data-toggle="dropdown"
  data-hover="dropdown"
  data-toggle="collapse"
  data-target="#info-extranet"
  data-original-title="CGU / RGPD"
  data-popup="tooltip"
  data-placement="top"
  data-original-title="Site internet de la FFSU"
  data-popup="tooltip"
  data-placement="top"
  data-original-title="Facebook"
  data-popup="tooltip"
  data-placement="top"
  data-original-title="Twitter"
  data-popup="tooltip"
  data-placement="top"
  data-original-title="Youtube"
  data-popup="tooltip"
  data-placement="top"
  data-original-title="Instagram"
  data-popup="tooltip"
  data-placement="top"
  data-toggle="tooltip"
  data-placement="left"


In [25]:
# Regarder l'existant - vérifier s'il y a déjà du code pour MySportU dans le projet
import os

# Chercher dans le projet existant
project_root = "/home/vincheetah/Documents/Travail/FFSU/PyCalendarClean/PyCalendar"

for root, dirs, files in os.walk(project_root):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                if 'mysportu' in content.lower():
                    print(f"Fichier avec mysportu: {filepath}")

Fichier avec mysportu: /home/vincheetah/Documents/Travail/FFSU/PyCalendarClean/PyCalendar/scripts/sync_mysportu.py


## Exploration des pages pour trouver les feuilles de match

Le site semble utiliser des pages classiques (pas de SPA). Cherchons les différentes sections pour trouver les rencontres.

In [26]:
# Explorer les différentes pages accessibles
pages_to_explore = [
    '/sportif/competitions',
    '/feuille-de-match',
    '/manifestations/calendrier-federal',
    '/sportif/extractions/rosters',
    '/extractions/sportif/joueurs',
]

for page in pages_to_explore:
    url = f"{BASE_URL}{page}"
    resp = session.get(url)
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # Chercher des tableaux
    tables = soup.find_all('table')
    
    # Chercher des liens spécifiques
    all_links = soup.find_all('a', href=True)
    specific_links = [l for l in all_links if '/rencontre' in l.get('href', '') or '/feuille' in l.get('href', '') or '/match' in l.get('href', '')]
    
    print(f"\n{'='*60}")
    print(f"Page: {page}")
    print(f"  Status: {resp.status_code}")
    print(f"  Tableaux trouvés: {len(tables)}")
    print(f"  Liens rencontres/matchs: {len(specific_links)}")
    
    for link in specific_links[:10]:
        print(f"    - {link.get('href')} - {link.get_text(strip=True)[:40]}")


Page: /sportif/competitions
  Status: 200
  Tableaux trouvés: 0
  Liens rencontres/matchs: 1
    - https://gestion.mysportu.com/feuille-de-match - Rencontres

Page: /feuille-de-match
  Status: 200
  Tableaux trouvés: 0
  Liens rencontres/matchs: 2
    - https://gestion.mysportu.com/feuille-de-match - Rencontres
    - https://gestion.mysportu.com/feuille-de-match - Rencontres

Page: /manifestations/calendrier-federal
  Status: 200
  Tableaux trouvés: 0
  Liens rencontres/matchs: 1
    - https://gestion.mysportu.com/feuille-de-match - Rencontres

Page: /sportif/extractions/rosters
  Status: 200
  Tableaux trouvés: 0
  Liens rencontres/matchs: 1
    - https://gestion.mysportu.com/feuille-de-match - Rencontres

Page: /extractions/sportif/joueurs
  Status: 200
  Tableaux trouvés: 0
  Liens rencontres/matchs: 1
    - https://gestion.mysportu.com/feuille-de-match - Rencontres


In [27]:
# Analyser en détail la page feuille-de-match
resp = session.get(f"{BASE_URL}/feuille-de-match")
html = resp.text

# Chercher les URLs dans le JavaScript
import re
js_urls = re.findall(r'["\']([^"\']*(?:api|rencontre|competition|feuille|match)[^"\']*)["\']', html, re.I)
print("URLs trouvées dans le JavaScript:")
for url in sorted(set(js_urls))[:50]:
    print(f"  {url}")

# Chercher les configurations JavaScript
config_patterns = [
    r'window\.__([A-Z_]+)\s*=\s*({[^}]+}|\'[^\']+\'|"[^"]+")',
    r'const\s+([a-zA-Z_]+)\s*=\s*({[^}]+})',
    r'let\s+([a-zA-Z_]+)\s*=\s*({[^}]+})',
]

print("\n\nConfigurations JavaScript trouvées:")
for pattern in config_patterns:
    matches = re.findall(pattern, html)
    for m in matches:
        print(f"  {m[0]}: {m[1][:100]}...")

URLs trouvées dans le JavaScript:
  
        ></app-feuille-de-match>
    </div>

    <script>
        window.avecSocket = 0;
        window.avecSocketPort = 6001;
    </script>
    <script src=
  />
    <title>FFSU - Feuille de match</title>
	<meta name=
  ></i>
                        <span>Rencontres</span>
                        </a>

            </li>
                                        <li class=
  >Rencontres</a>

                        </div>
                        
                    </div>

                    
                    <div class=
  Rencontres
  feuilleDeMatch
  https://fonts.googleapis.com/css?family=Roboto:400,300,100,500,700,900
  https://gestion.mysportu.com/extractions/arbitrage/designations-par-competition-lieu-personne
  https://gestion.mysportu.com/feuille-de-match
  https://gestion.mysportu.com/sportif/competitions
  https://gestion.mysportu.com/sportif/validations/competitions
  {&quot;id&quot;:10526,&quot;est_federation&quot;:false,&quot;role&qu

In [28]:
# Chercher spécifiquement les appels AJAX/Livewire
resp = session.get(f"{BASE_URL}/feuille-de-match")
html = resp.text

# Chercher des composants Livewire
livewire_patterns = [
    r'wire:snapshot="([^"]+)"',
    r'wire:effects="([^"]+)"',
    r'wire:id="([^"]+)"',
    r'@livewire\(\'([^\']+)\'',
]

print("Composants Livewire trouvés:")
for pattern in livewire_patterns:
    matches = re.findall(pattern, html)
    for m in matches[:5]:
        if len(m) > 100:
            print(f"  {m[:100]}...")
        else:
            print(f"  {m}")

Composants Livewire trouvés:


In [29]:
# Afficher une partie du HTML pour mieux comprendre la structure
# Chercher les zones de contenu
soup = BeautifulSoup(html, 'html.parser')

# Chercher les classes CSS utilisées pour le contenu
content_area = soup.find('div', class_='content')
if content_area:
    print("=== Zone de contenu ===")
    print(content_area.prettify()[:3000])
else:
    # Chercher d'autres conteneurs
    main_content = soup.find('main') or soup.find('div', {'id': 'app'}) or soup.find('div', {'role': 'main'})
    if main_content:
        print("=== Contenu principal ===")
        print(main_content.prettify()[:3000])
    else:
        # Afficher le body
        body = soup.find('body')
        if body:
            print("=== Body (extrait) ===")
            print(str(body)[:4000])

=== Zone de contenu ===
<div class="content">
 <div id="flashbag_container">
  <flashbag ref="flashbag">
  </flashbag>
 </div>
 <script>
  const flashbag = new Vue({
        el: '#flashbag_container',
    });

    var messages_types = {};





    flashbag.$refs.flashbag.nouveaux_messages(messages_types);
 </script>
 <div id="feuilleDeMatch">
  <app-feuille-de-match :config="{&quot;signatures_password&quot;:&quot;unique&quot;,&quot;horaire_vestiaire&quot;:false,&quot;avec_couleur_domicile&quot;:true,&quot;stysteme_ligne&quot;:false,&quot;lignes_par_default&quot;:[],&quot;ajout_ligne&quot;:false,&quot;deplacement_ligne&quot;:true,&quot;libelle_ligne&quot;:&quot;Ligne&quot;,&quot;staff_peut_jouer&quot;:true,&quot;officiel_peut_jouer&quot;:false,&quot;staff_peut_arbitrer&quot;:false,&quot;joueur_prete_peut_joueur_sur_deux_structures&quot;:true,&quot;service_regles_federation&quot;:&quot;App\\Extranet\\FeuilleDeMatch\\live\\src\\Services\\FeuilleDeMatchServiceValidation&quot;,&quot;code_me

In [30]:
# Chercher les scripts externes chargés
scripts_ext = soup.find_all('script', src=True)
print("Scripts externes chargés:")
for script in scripts_ext:
    src = script.get('src')
    print(f"  {src}")

Scripts externes chargés:
  /js/vendor.js?id=461314863c1c08b8f668
  /js/app.js?id=d41d8cd98f00b204e980
  /elicence-feuille-de-match/js/mains.js?id=dcadf14ff59934c8a33c


In [31]:
# Récupérer le script des feuilles de match pour analyser les endpoints
js_url = f"{BASE_URL}/elicence-feuille-de-match/js/mains.js"
resp_js = session.get(js_url)
print(f"Status: {resp_js.status_code}")
print(f"Taille: {len(resp_js.text)} caractères")

# Chercher les endpoints API dans le JavaScript
api_patterns = [
    r'["\']/?api/[^"\']+["\']',
    r'["\']/?elicence[^"\']*["\']',
    r'["\']/?feuille[^"\']*["\']',
    r'["\']/?rencontre[^"\']*["\']',
    r'\.get\(["\']([^"\']+)["\']',
    r'\.post\(["\']([^"\']+)["\']',
    r'fetch\(["\']([^"\']+)["\']',
    r'url:\s*["\']([^"\']+)["\']',
]

js_content = resp_js.text
found_endpoints = set()
for pattern in api_patterns:
    matches = re.findall(pattern, js_content)
    found_endpoints.update(matches)

print(f"\nEndpoints trouvés ({len(found_endpoints)}):")
for ep in sorted(found_endpoints)[:50]:
    print(f"  {ep}")

Status: 200
Taille: 1458084 caractères

Endpoints trouvés (26):
  "/elicence-feuille-de-match/images/empty-blason.png"
  "/rencontre/:rencontreId"
  "/rencontre/:rencontreId/controle-participants"
  "/rencontre/:rencontreId/equipe/:id"
  "/rencontre/:rencontreId/equipe/:id/attribution"
  "/rencontre/:rencontreId/live"
  "/rencontre/:rencontreId/live/:name"
  "/rencontre/:rencontreId/logs"
  "/rencontre/:rencontreId/officiels/attribution"
  "/rencontre/:rencontreId/penalites"
  "/rencontre/:rencontreId/penalites/:penalite"
  "/rencontre/:rencontreId/penalites/:penalite/:penaliteId"
  "/rencontre/:rencontreId/penalites/:penalite/nouveau"
  "/rencontre/:rencontreId/reserves"
  "/rencontre/:rencontreId/reset"
  "/rencontre/:rencontreId/signatures-apres-match"
  "/rencontre/:rencontreId/signatures-avant-match"
  "/rencontre/:rencontreId/staff/:id/attribution"
  "/rencontre/:rencontreId/upload"
  "rencontre"
  "rencontre-"
  "rencontre-item"
  "rencontre."
  "rencontreId"
  "rencontreItem"
 

In [32]:
# Chercher plus d'endpoints dans le script
more_patterns = [
    r'baseURL["\s:]+["\']([^"\']+)["\']',
    r'axios\.create\([^)]*baseURL[^)]+\)',
    r'API_[A-Z_]+\s*[=:]\s*["\']([^"\']+)["\']',
    r'["\']?url["\']?\s*:\s*["\']([^"\']+)["\']',
]

print("Recherche de configurations d'API...")
for pattern in more_patterns:
    matches = re.findall(pattern, js_content)
    if matches:
        print(f"\nPattern: {pattern}")
        for m in set(matches)[:10]:
            print(f"  {m}")

Recherche de configurations d'API...


In [33]:
# Chercher l'API elicence-feuille-de-match - tester différents endpoints
api_base = f"{BASE_URL}/elicence-feuille-de-match/api"

test_endpoints = [
    '/rencontres',
    '/rencontre',
    '/matches',
    '/competitions',
    '/journees',
    '/manifestations',
    '',
]

print("Test des endpoints API feuille-de-match:")
for ep in test_endpoints:
    url = f"{api_base}{ep}"
    resp = session.get(url, headers={'Accept': 'application/json'})
    print(f"\n{ep or '/'}: {resp.status_code}")
    if resp.status_code == 200:
        try:
            data = resp.json()
            print(f"  JSON: {str(data)[:300]}")
        except:
            print(f"  HTML: {resp.text[:200]}")

Test des endpoints API feuille-de-match:

/rencontres: 404

/rencontre: 404

/matches: 404

/competitions: 404

/journees: 404

/manifestations: 404

/: 404


In [34]:
# Tester avec le chemin sans /api
test_paths = [
    '/elicence-feuille-de-match/rencontres',
    '/elicence-feuille-de-match/rencontre',
    '/elicence-feuille-de-match',
    '/api/rencontres',
    '/api/rencontre',
    '/api/competitions',
    '/api/v1/rencontres',
]

print("Test de différents chemins:")
for path in test_paths:
    url = f"{BASE_URL}{path}"
    resp = session.get(url, headers={'Accept': 'application/json'})
    print(f"\n{path}: {resp.status_code}")
    if resp.status_code == 200:
        content_type = resp.headers.get('Content-Type', '')
        print(f"  Content-Type: {content_type}")
        if 'json' in content_type:
            try:
                data = resp.json()
                print(f"  JSON: {str(data)[:300]}")
            except:
                pass
        else:
            print(f"  HTML: {resp.text[:200]}")

Test de différents chemins:

/elicence-feuille-de-match/rencontres: 404

/elicence-feuille-de-match/rencontre: 404

/elicence-feuille-de-match: 404

/api/rencontres: 404

/api/rencontre: 404

/api/competitions: 404

/api/v1/rencontres: 404


In [35]:
# Regarder dans la page HTML pour trouver des données de rencontres
# Chercher le SPA state ou les données initiales
resp = session.get(f"{BASE_URL}/feuille-de-match")
html = resp.text

# Chercher les données JSON embarquées dans la page
json_data_patterns = [
    r'window\.rencontres\s*=\s*(\[[^\]]*\])',
    r'window\.matches\s*=\s*(\[[^\]]*\])',
    r'data\s*:\s*(\{[^}]+rencontre[^}]+\})',
    r'rencontres\s*:\s*(\[[^\]]+\])',
    r'<script[^>]*id="[^"]*data[^"]*"[^>]*>([^<]+)</script>',
]

print("Recherche de données JSON embarquées...")
for pattern in json_data_patterns:
    matches = re.findall(pattern, html, re.DOTALL)
    if matches:
        print(f"\nPattern trouvé: {pattern[:50]}...")
        for m in matches[:3]:
            print(f"  {str(m)[:300]}...")

Recherche de données JSON embarquées...


In [36]:
# Essayer de récupérer le contenu de la page feuille-de-match
# Regarder s'il y a une iframe ou un composant Vue/React
soup = BeautifulSoup(html, 'html.parser')

# Chercher les iframes
iframes = soup.find_all('iframe')
print(f"Iframes trouvées: {len(iframes)}")
for iframe in iframes:
    print(f"  src: {iframe.get('src')}")

# Chercher les data-* attributes avec des données JSON
data_attrs_all = []
for el in soup.find_all(True):
    for attr, value in el.attrs.items():
        if attr.startswith('data-') and len(str(value)) > 50:
            data_attrs_all.append((attr, value))

print(f"\nData attributes avec données longues: {len(data_attrs_all)}")
for attr, value in data_attrs_all[:5]:
    print(f"  {attr}: {str(value)[:200]}...")

Iframes trouvées: 0

Data attributes avec données longues: 0


In [37]:
# Regarder les divs avec des id ou classes spécifiques
print("Éléments avec des id pertinents:")
for el in soup.find_all(id=True):
    el_id = el.get('id')
    if any(k in el_id.lower() for k in ['app', 'root', 'main', 'content', 'vue', 'react', 'rencontre', 'match']):
        print(f"  <{el.name} id='{el_id}'>")
        # Afficher les attributs
        for attr, val in el.attrs.items():
            if attr != 'id':
                val_str = str(val)[:100]
                print(f"    {attr}: {val_str}")

# Regarder spécifiquement l'élément où l'app est montée
print("\n\nDiv avec classes spécifiques:")
for el in soup.find_all(class_=True):
    classes = el.get('class', [])
    if any(c for c in classes if any(k in c.lower() for k in ['feuille', 'rencontre', 'match', 'app-'])):
        print(f"  <{el.name} class='{' '.join(classes)}'>")

Éléments avec des id pertinents:
  <div id='feuilleDeMatch'>


Div avec classes spécifiques:


In [38]:
# Regarder le contenu autour de feuilleDeMatch et les scripts inline
feuille_div = soup.find(id='feuilleDeMatch')
if feuille_div:
    print("Contenu du div feuilleDeMatch:")
    print(feuille_div.prettify()[:1000])
    
    # Chercher les data-* attributes
    print("\nAttributs:")
    for attr, val in feuille_div.attrs.items():
        print(f"  {attr}: {str(val)[:200]}")

# Afficher les scripts inline pour voir comment l'app est initialisée
print("\n\nScripts inline:")
for script in soup.find_all('script'):
    if script.string and ('feuille' in script.string.lower() or 'rencontre' in script.string.lower() or 'vue' in script.string.lower() or 'mount' in script.string.lower()):
        print(f"\n--- Script ---")
        print(script.string[:1000])

Contenu du div feuilleDeMatch:
<div id="feuilleDeMatch">
 <app-feuille-de-match :config="{&quot;signatures_password&quot;:&quot;unique&quot;,&quot;horaire_vestiaire&quot;:false,&quot;avec_couleur_domicile&quot;:true,&quot;stysteme_ligne&quot;:false,&quot;lignes_par_default&quot;:[],&quot;ajout_ligne&quot;:false,&quot;deplacement_ligne&quot;:true,&quot;libelle_ligne&quot;:&quot;Ligne&quot;,&quot;staff_peut_jouer&quot;:true,&quot;officiel_peut_jouer&quot;:false,&quot;staff_peut_arbitrer&quot;:false,&quot;joueur_prete_peut_joueur_sur_deux_structures&quot;:true,&quot;service_regles_federation&quot;:&quot;App\\Extranet\\FeuilleDeMatch\\live\\src\\Services\\FeuilleDeMatchServiceValidation&quot;,&quot;code_medecin_universel&quot;:&quot;HnnVCSK1deABjkEL&quot;,&quot;code_signataire_universel&quot;:&quot;HnnVCSK1deABjkEL&quot;,&quot;avec_socket&quot;:false,&quot;equipe_adverse_visible&quot;:{&quot;equipe_adverse_valide&quot;:true,&quot;equipe_initial_valide&quot;:true},&quot;notification_equipe&

In [39]:
# Extraire la configuration complète du composant
import html

feuille_div = soup.find(id='feuilleDeMatch')
app_component = feuille_div.find('app-feuille-de-match')

if app_component:
    # Récupérer l'attribut :config
    config_raw = app_component.get(':config')
    if config_raw:
        # Décoder les entités HTML
        config_decoded = html.unescape(config_raw)
        print("Configuration décodée:")
        try:
            config = json.loads(config_decoded)
            print(json.dumps(config, indent=2, ensure_ascii=False)[:2000])
        except json.JSONDecodeError as e:
            print(f"Erreur JSON: {e}")
            print(config_decoded[:1000])

Configuration décodée:
{
  "signatures_password": "unique",
  "horaire_vestiaire": false,
  "avec_couleur_domicile": true,
  "stysteme_ligne": false,
  "lignes_par_default": [],
  "ajout_ligne": false,
  "deplacement_ligne": true,
  "libelle_ligne": "Ligne",
  "staff_peut_jouer": true,
  "officiel_peut_jouer": false,
  "staff_peut_arbitrer": false,
  "joueur_prete_peut_joueur_sur_deux_structures": true,
  "service_regles_federation": "App\\Extranet\\FeuilleDeMatch\\live\\src\\Services\\FeuilleDeMatchServiceValidation",
  "code_medecin_universel": "HnnVCSK1deABjkEL",
  "code_signataire_universel": "HnnVCSK1deABjkEL",
  "avec_socket": false,
  "equipe_adverse_visible": {
    "equipe_adverse_valide": true,
    "equipe_initial_valide": true
  },
  "notification_equipe": {
    "active": true,
    "correspondant_equipe": true
  },
  "notification_report": {
    "active": true
  },
  "notification_modification_rencontre": {
    "active": false
  },
  "notification_reserves": {
    "active": f

In [40]:
# Chercher les endpoints dans le script JavaScript principal
# Regarder comment les rencontres sont chargées

# Télécharger et analyser le script mains.js
js_url = f"{BASE_URL}/elicence-feuille-de-match/js/mains.js"
resp_js = session.get(js_url)
js_content = resp_js.text

# Chercher les patterns d'appels API
api_call_patterns = [
    r'axios\.(get|post)\(["\']([^"\']+)["\']',
    r'fetch\(["\']([^"\']+)["\']',
    r'this\.\$http\.(get|post)\(["\']([^"\']+)["\']',
    r'\.get\(["\']([^"\']+rencontre[^"\']*)["\']',
    r'\.post\(["\']([^"\']+rencontre[^"\']*)["\']',
]

print("Appels API trouvés dans mains.js:")
all_api_calls = []
for pattern in api_call_patterns:
    matches = re.findall(pattern, js_content)
    all_api_calls.extend(matches)

# Dédupliquer et afficher
unique_calls = set()
for m in all_api_calls:
    if isinstance(m, tuple):
        unique_calls.add(m[-1] if isinstance(m[-1], str) else str(m))
    else:
        unique_calls.add(m)

for call in sorted(unique_calls)[:30]:
    if 'rencontre' in call.lower() or 'api' in call.lower() or call.startswith('/'):
        print(f"  {call}")

Appels API trouvés dans mains.js:


In [41]:
# Chercher des patterns plus génériques dans le JS
patterns = [
    r'"/[a-zA-Z-]+/[a-zA-Z-]+"',
    r"'/[a-zA-Z-]+/[a-zA-Z-]+'",
    r'baseURL\s*[=:]\s*["\']([^"\']+)["\']',
    r'url\s*[=:]\s*["\']([^"\']+)["\']',
]

print("URLs trouvées dans le JS:")
found_urls = set()
for pattern in patterns:
    matches = re.findall(pattern, js_content)
    for m in matches:
        if '/rencontre' in m.lower() or '/api' in m.lower() or '/elicence' in m.lower():
            found_urls.add(m)

for url in sorted(found_urls):
    print(f"  {url}")

URLs trouvées dans le JS:


In [42]:
# Regarder un extrait du JS autour des mots-clés importants
import re

# Trouver le contexte autour de "rencontre" dans le JS
for match in re.finditer(r'.{100}rencontre.{100}', js_content, re.IGNORECASE):
    print(match.group())
    print("---")

 strict";n.r(t);var r=n("f+W4"),i={name:"index",props:{avecFiltres:{type:Boolean}},components:{listeRencontre:r.a},data:function(){return{}},methods:{},computed:{}},o=n("KHd+"),a=Object(o.a)(i,(function(){var 
---
nstanceof t))throw new TypeError("Cannot call a class as a function")}(this,e),this._routes=t,this._rencontreId=n,this._equipeId=r},(t=[{key:"routes",get:function(){return this._routes},set:function(e){this._r
---
>0&&void 0!==i[0]&&i[0],n=i.length>1&&void 0!==i[1]&&i[1],r=this.routes.validation_joueurs.replace("rencontre_id",this.rencontreId),e.next=5,axios.post(r,{equipe_id:this.equipeId,ignore_warnings:t,sans_control
---
(function(e){for(;;)switch(e.prev=e.next){case 0:return n=this.routes.devalidation_joueurs.replace("rencontre_id",this.rencontreId),e.next=3,axios.post(n,{equipe_id:t}).then((function(e){return e})).catch((fun
---
v().wrap((function(e){for(;;)switch(e.prev=e.next){case 0:return n=this.routes.ajout_ligne.replace("rencontre_id",this.rencontreId),e.next=3,axi

In [43]:
# Regarder les données passées directement en prop sur le composant
# Récupérer tous les attributs du composant app-feuille-de-match
resp = session.get(f"{BASE_URL}/feuille-de-match")
soup = BeautifulSoup(resp.text, 'html.parser')
feuille_div = soup.find(id='feuilleDeMatch')
app_component = feuille_div.find('app-feuille-de-match')

if app_component:
    print("Tous les attributs du composant:")
    for attr, val in app_component.attrs.items():
        if attr.startswith(':'):
            print(f"\n{attr}:")
            decoded = html.unescape(str(val))
            print(decoded[:500] + "..." if len(decoded) > 500 else decoded)

Tous les attributs du composant:

:routes:
{"liste_rencontres":"https:\/\/gestion.mysportu.com\/feuille-de-match\/rencontres","export_rencontres":"https:\/\/gestion.mysportu.com\/feuille-de-match\/rencontres\/export","liste_competitions":"https:\/\/gestion.mysportu.com\/feuille-de-match\/ajax\/competitions","liste_structures":"https:\/\/gestion.mysportu.com\/feuille-de-match\/ajax\/structures","liste_disciplines":"https:\/\/gestion.mysportu.com\/feuille-de-match\/ajax\/disciplines","rencontre":"https:\/\/gestion.mysportu.com\/feuille-de-...

:user:
{"id":10526,"est_federation":false,"role":null,"est_officiel":false,"gestion_equipes_ids":[98,99,100,101,167,168,169,253,254,255,256,257,258,259,260,368,412,418,492,715,743,748,757,758,784,789,818,819,821,823,824,825,826,827,829,830,831,832,833,835,836,841,842,843,844,845,846,848,849,907,920,921,922,923,924,949,950,951,952,953,954,955,956,957,958,959,960,963,964,967,968,969,970,971,972,973,974,975,998,999,1000,1002,1003,1006,1007,1008,1010,1

## 🎯 Endpoints API découverts !

Les routes API pour les feuilles de match sont :
- `liste_rencontres`: `/feuille-de-match/rencontres`
- `export_rencontres`: `/feuille-de-match/rencontres/export`
- `liste_competitions`: `/feuille-de-match/ajax/competitions`
- `rencontre`: `/feuille-de-match/rencontre/{id}`

In [44]:
# Extraire toutes les routes de l'attribut :routes
routes_raw = app_component.get(':routes')
routes_decoded = html.unescape(routes_raw)
routes = json.loads(routes_decoded)

print("Routes API disponibles:")
for name, url in routes.items():
    print(f"  {name}: {url}")

Routes API disponibles:
  liste_rencontres: https://gestion.mysportu.com/feuille-de-match/rencontres
  export_rencontres: https://gestion.mysportu.com/feuille-de-match/rencontres/export
  liste_competitions: https://gestion.mysportu.com/feuille-de-match/ajax/competitions
  liste_structures: https://gestion.mysportu.com/feuille-de-match/ajax/structures
  liste_disciplines: https://gestion.mysportu.com/feuille-de-match/ajax/disciplines
  rencontre: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id
  participants_rencontres: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/participants
  dupliquer_joueurs: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/dupliquer/joueurs
  dupliquer_staffs: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/dupliquer/staffs
  update_rencontre: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id
  update_rencontre_detail: https://gestion.mysportu.com/feuille-de-m

In [45]:
# Tester l'endpoint liste_rencontres
rencontres_url = routes.get('liste_rencontres')
print(f"Test de: {rencontres_url}")

resp = session.get(rencontres_url, headers={
    'Accept': 'application/json',
    'X-Requested-With': 'XMLHttpRequest'
})
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type')}")

if resp.status_code == 200:
    try:
        data = resp.json()
        print(f"\nType de données: {type(data)}")
        if isinstance(data, dict):
            print(f"Clés: {list(data.keys())}")
            for key in list(data.keys())[:5]:
                print(f"  {key}: {str(data[key])[:200]}")
        elif isinstance(data, list):
            print(f"Nombre d'éléments: {len(data)}")
            if data:
                print(f"Premier élément: {data[0]}")
    except:
        print(f"Pas de JSON, contenu HTML: {resp.text[:500]}")

Test de: https://gestion.mysportu.com/feuille-de-match/rencontres
Status: 200
Content-Type: application/json

Type de données: <class 'dict'>
Clés: ['data', 'links', 'meta']
  data: [{'id': 4070, 'receveur': {'id': 412, 'participant_id': None, 'libelle': '069069006 - Association Sportive Université de Lyon', 'libelle_court': 'AS UD Lyon', 'logo': 'https://gestion.mysportu.com/sto
  links: {'first': 'https://gestion.mysportu.com/feuille-de-match/rencontres?page=1', 'last': 'https://gestion.mysportu.com/feuille-de-match/rencontres?page=39', 'prev': None, 'next': 'https://gestion.mysportu
  meta: {'current_page': 1, 'from': 1, 'last_page': 39, 'links': [{'url': None, 'label': '&laquo; Précédent', 'page': None, 'active': False}, {'url': 'https://gestion.mysportu.com/feuille-de-match/rencontres?


In [46]:
# Récupérer les compétitions disponibles pour filtrer
competitions_url = routes.get('liste_competitions')
print(f"Test de: {competitions_url}")

resp = session.get(competitions_url, headers={
    'Accept': 'application/json',
    'X-Requested-With': 'XMLHttpRequest'
})
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    competitions = resp.json()
    print(f"\nNombre de compétitions: {len(competitions)}")
    
    # Chercher les compétitions de volley
    volley_comps = [c for c in competitions if 'volley' in str(c).lower()]
    print(f"\nCompétitions de Volley ({len(volley_comps)}):")
    for comp in volley_comps:
        print(f"  {comp}")

Test de: https://gestion.mysportu.com/feuille-de-match/ajax/competitions
Status: 200

Nombre de compétitions: 750

Compétitions de Volley (123):
  {'id': 211, 'libelle': 'ANGERS District VOLLEY 6X6', 'structure_id': 122, 'saison': 2026, 'etat': 'A', 'engagements_clos': 0, 'calendrier_id': 403, 'sexe': 'FM', 'entente': 0, 'type_id': None, 'type_engagement': 'ENFA', 'sportif_compte_bancaire_id': None, 'parametres': {'paiement_inscription': True, 'tag_competition': None, 'type_paiement': {'code': 'PIME', 'libelle': 'Paiement Immediat'}, 'paiement_multiple': [], 'montant_inscription': 0, 'structure_referente': None, 'montant_droit_arbitrage': 0, 'mouvement_comptable': None, 'compta_analytique': None, 'prelevement_forfait': 0, 'prelevement_forfait_saison': 0, 'rib_different': False, 'nom_banque': None, 'code_etablissement': None, 'code_guichet': None, 'numero_banque': None, 'cle_rib': None, 'iban': None, 'bic': None, 'calendrier_jours_disponible': [], 'calendrier_heure_defaut': None, 'utili

In [47]:
# Afficher uniquement les compétitions de volley 6x6
for comp in competitions:
    name = comp.get('libelle', '') or comp.get('nom', '') or str(comp)
    if 'volley' in name.lower() and '6' in name:
        print(f"ID: {comp.get('id')} - {name}")

ID: 211 - ANGERS District VOLLEY 6X6
ID: 704 - BEACH VOLLEY TOURNOI FEMININ DU 02 04 2026
ID: 707 - BEACH VOLLEY TOURNOI FEMININ DU 09 04 2026
ID: 706 - BEACH VOLLEY TOURNOI MASCULIN DU 02 04 2026
ID: 708 - BEACH VOLLEY TOURNOI MASCULIN DU 09 04 2026
ID: 779 - CF des IUT de volley 6X6 Masculin
ID: 459 - CHPT ACAD AIX-MARSEILLE VOLLEY 6X6 / FEMININ
ID: 460 - CHPT ACAD AIX-MARSEILLE VOLLEY 6X6 / MASCULIN
ID: 365 - Champ Acad volley 6X6 Féminin
ID: 364 - Champ Acad volley 6X6 Masculin
ID: 373 - Champ Acad volley 6X6 Mixte
ID: 317 - Championnat Acad Nice-Toulon Volley Ball 6x6/Féminin
ID: 316 - Championnat Acad Nice-Toulon Volley Ball 6x6/Masculin
ID: 318 - Championnat Acad Nice-Toulon Volley Ball 6x6/Mixte
ID: 126 - Championnat Académique Ecole de volley 6X6 Féminin
ID: 118 - Championnat Académique Ecole de volley 6X6 Masculin
ID: 723 - Championnat Académique Universitaire de volley 6X6 Féminin - Limoges
ID: 586 - Championnat Académique Universitaire de volley 6X6 Masculin - Limoges
ID: 3

In [48]:
# Regarder la structure d'une compétition
print("Structure d'une compétition:")
if competitions:
    print(json.dumps(competitions[0], indent=2, ensure_ascii=False))

Structure d'une compétition:
{
  "id": 752,
  "libelle": "Championnat de France Universitaire de rugby 10 Masculin 2026",
  "structure_id": 1,
  "saison": 2026,
  "etat": "A",
  "engagements_clos": 0,
  "calendrier_id": 4700,
  "sexe": "M",
  "entente": 0,
  "type_id": null,
  "type_engagement": "TOUS",
  "sportif_compte_bancaire_id": null,
  "parametres": {
    "paiement_inscription": true,
    "tag_competition": null,
    "type_paiement": {
      "code": "PIME",
      "libelle": "Paiement Immediat"
    },
    "paiement_multiple": [],
    "montant_inscription": 0,
    "structure_referente": null,
    "montant_droit_arbitrage": 0,
    "mouvement_comptable": null,
    "compta_analytique": null,
    "prelevement_forfait": 0,
    "prelevement_forfait_saison": 0,
    "rib_different": false,
    "nom_banque": null,
    "code_etablissement": null,
    "code_guichet": null,
    "numero_banque": null,
    "cle_rib": null,
    "iban": null,
    "bic": null,
    "calendrier_jours_disponible": []

In [49]:
# Récupérer les rencontres avec des filtres pour le 5 février et volley
# Tester les paramètres de filtrage

# D'abord, récupérer les données du premier match pour voir la structure
rencontres_url = routes.get('liste_rencontres')
resp = session.get(rencontres_url, headers={
    'Accept': 'application/json',
    'X-Requested-With': 'XMLHttpRequest'
})
data = resp.json()

# Afficher la structure d'une rencontre
if data['data']:
    print("Structure d'une rencontre:")
    print(json.dumps(data['data'][0], indent=2, ensure_ascii=False)[:3000])

Structure d'une rencontre:
{
  "id": 4070,
  "receveur": {
    "id": 412,
    "participant_id": null,
    "libelle": "069069006 - Association Sportive Université de Lyon",
    "libelle_court": "AS UD Lyon",
    "logo": "https://gestion.mysportu.com/storage/logos_equipes/212/cDSughNMS6d6STb7Cpl9Knog7wGhHe2nFoYWjkLr.png",
    "club": {
      "id": 212,
      "nom": "UDL - UTE LYON 2",
      "type": "CLU",
      "code": "069069006",
      "color": "primary",
      "TypeColorLabel": "primary",
      "styled": "<span class=\"text-primary\">\n                \n        <span class=\"badge badge-primary position-relative mr-1\"> 069069006</span>\n        UDL - UTE LYON 2 \n</span>\n",
      "type_libelle": "AS"
    },
    "equipement_domcile_visuel": null,
    "equipement_domicile": {
      "Maillot": "#ffffff",
      "Casque": "#16a5de"
    },
    "equipement_exterieur_visuel": null,
    "equipement_exterieur": {
      "Maillot": "#000000",
      "Casque": "#16a5de"
    },
    "structuresIdsS

In [50]:
# Tester les filtres disponibles pour l'API
# Essayons avec des paramètres GET

# Filtrer par date (5 février 2026)
date_filter = "2026-02-05"

# Tester différentes combinaisons de filtres
test_params = [
    {'date': date_filter},
    {'date_debut': date_filter, 'date_fin': date_filter},
    {'search': 'volley'},
    {'discipline': 'D_VOLLEY_6X6'},
]

for params in test_params:
    resp = session.get(rencontres_url, params=params, headers={
        'Accept': 'application/json',
        'X-Requested-With': 'XMLHttpRequest'
    })
    data = resp.json()
    print(f"Params: {params}")
    print(f"  Résultats: {len(data.get('data', []))} / Total pages: {data.get('meta', {}).get('last_page', 'N/A')}")
    if data.get('data'):
        first = data['data'][0]
        print(f"  Premier match: {first.get('infosRencontre', {}).get('competition_libelle', 'N/A')[:60]}")
    print()

Params: {'date': '2026-02-05'}
  Résultats: 50 / Total pages: 39
  Premier match: Championnat de France Universitaire de football Masculin N1

Params: {'date_debut': '2026-02-05', 'date_fin': '2026-02-05'}
  Résultats: 50 / Total pages: 39
  Premier match: Championnat de France Universitaire de football Masculin N1

Params: {'search': 'volley'}
  Résultats: 50 / Total pages: 39
  Premier match: Championnat de France Universitaire de football Masculin N1

Params: {'discipline': 'D_VOLLEY_6X6'}
  Résultats: 50 / Total pages: 39
  Premier match: Championnat de France Universitaire de football Masculin N1



In [51]:
# Parcourir toutes les pages pour récupérer toutes les rencontres
# et filtrer côté Python pour les matchs de volley du 5 février

all_rencontres = []
page = 1
max_pages = 40

while page <= max_pages:
    resp = session.get(f"{rencontres_url}?page={page}", headers={
        'Accept': 'application/json',
        'X-Requested-With': 'XMLHttpRequest'
    })
    data = resp.json()
    
    rencontres = data.get('data', [])
    all_rencontres.extend(rencontres)
    
    if page % 10 == 0:
        print(f"Page {page}/{data.get('meta', {}).get('last_page', 'N/A')} - Total: {len(all_rencontres)}")
    
    if not data.get('links', {}).get('next'):
        break
    page += 1

print(f"\nTotal rencontres récupérées: {len(all_rencontres)}")

Page 10/39 - Total: 500
Page 20/39 - Total: 1000
Page 30/39 - Total: 1500

Total rencontres récupérées: 1935


In [52]:
# Analyser les rencontres - trouver les compétitions uniques
competitions_uniques = set()
for r in all_rencontres:
    comp = r.get('infosRencontre', {}).get('competition_libelle', 'N/A')
    competitions_uniques.add(comp)

print(f"Compétitions uniques ({len(competitions_uniques)}):")
for comp in sorted(competitions_uniques):
    if 'volley' in comp.lower():
        print(f"  🏐 {comp}")

Compétitions uniques (106):
  🏐 Championnat Inter-Ligues des IUT de volley Féminin
  🏐 Championnat Inter-Ligues des IUT de volley Masculin
  🏐 Coupe de France des ESC Volley 6X6 Féminin
  🏐 Coupe de France des ESC Volley 6X6 Masculin
  🏐 LYON VBF PH2 - Championnat Académique de Volley Féminin
  🏐 LYON VBM PH2 - Championnat Académique Volley Masculin
  🏐 STE VBM - Championnat local Universitaire de volley 6X6 Masculin
  🏐 STE VBMX - Championnat local Universitaire de volley 6X6 Mixte


In [53]:
# Filtrer les rencontres de volley Lyon (PH2) du 5 février 2026
from datetime import datetime

target_date = "2026-02-05"

volley_lyon_matches = []
for r in all_rencontres:
    comp = r.get('infosRencontre', {}).get('competition_libelle', '')
    if 'LYON' in comp and ('VBF' in comp or 'VBM' in comp) and 'PH2' in comp:
        # Récupérer la date du match
        date_str = r.get('infosRencontre', {}).get('date_debut') or r.get('infosRencontre', {}).get('date')
        
        # Vérifier si c'est le 5 février
        if date_str and target_date in str(date_str):
            volley_lyon_matches.append(r)

print(f"Matchs de Volley Lyon PH2 le {target_date}: {len(volley_lyon_matches)}")

# Afficher les détails
for match in volley_lyon_matches:
    info = match.get('infosRencontre', {})
    receveur = match.get('receveur', {}).get('libelle', 'N/A')
    visiteur = match.get('visiteur', {}).get('libelle', 'N/A')
    date = info.get('date_debut') or info.get('date')
    heure = info.get('heure_debut') or info.get('heure')
    lieu = info.get('lieu_libelle') or info.get('lieu')
    comp = info.get('competition_libelle', '')
    
    print(f"\n🏐 {receveur[:30]} vs {visiteur[:30]}")
    print(f"   Date: {date} {heure}")
    print(f"   Lieu: {lieu}")
    print(f"   Competition: {comp}")

Matchs de Volley Lyon PH2 le 2026-02-05: 0


In [54]:
# Voir toutes les dates des matchs de volley Lyon PH2
volley_lyon_all = []
for r in all_rencontres:
    comp = r.get('infosRencontre', {}).get('competition_libelle', '')
    if 'LYON' in comp and ('VBF' in comp or 'VBM' in comp) and 'PH2' in comp:
        volley_lyon_all.append(r)

print(f"Total matchs Volley Lyon PH2: {len(volley_lyon_all)}")

# Lister les dates uniques
dates_uniques = set()
for match in volley_lyon_all:
    info = match.get('infosRencontre', {})
    date = info.get('date_debut') or info.get('date') or 'N/A'
    dates_uniques.add(str(date)[:10])

print(f"\nDates disponibles:")
for d in sorted(dates_uniques):
    print(f"  {d}")

Total matchs Volley Lyon PH2: 221

Dates disponibles:
  N/A


In [55]:
# Examiner la structure complète d'un match volley Lyon pour trouver la date
if volley_lyon_all:
    print("Structure complète d'un match Volley Lyon PH2:")
    print(json.dumps(volley_lyon_all[0], indent=2, ensure_ascii=False))

Structure complète d'un match Volley Lyon PH2:
{
  "id": 20181,
  "receveur": {
    "id": 970,
    "participant_id": null,
    "libelle": "069069031 - LYON 1 (1)",
    "libelle_court": "LYON 1 (1)",
    "logo": "https://gestion.mysportu.com/storage/logos_equipes/696/1KVz6Kjw9x7U4kJ7hbpKnoqXw9045ZGiyxivAHd1.jpg",
    "club": {
      "id": 696,
      "nom": "UDL - UTE LYON 1",
      "type": "CLU",
      "code": "069069031",
      "color": "primary",
      "TypeColorLabel": "primary",
      "styled": "<span class=\"text-primary\">\n                \n        <span class=\"badge badge-primary position-relative mr-1\"> 069069031</span>\n        UDL - UTE LYON 1 \n</span>\n",
      "type_libelle": "AS"
    },
    "equipement_domcile_visuel": null,
    "equipement_domicile": {
      "Maillot": "#16a5de",
      "Casque": "#16a5de"
    },
    "equipement_exterieur_visuel": null,
    "equipement_exterieur": {
      "Maillot": "#16a5de",
      "Casque": "#16a5de"
    },
    "structuresIdsScope": [

In [56]:
# Chercher récursivement toutes les clés contenant "date" ou "heure"
def find_date_keys(obj, prefix=""):
    results = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            full_key = f"{prefix}.{key}" if prefix else key
            if 'date' in key.lower() or 'heure' in key.lower() or 'time' in key.lower():
                results.append((full_key, value))
            results.extend(find_date_keys(value, full_key))
    elif isinstance(obj, list) and obj:
        results.extend(find_date_keys(obj[0], f"{prefix}[0]"))
    return results

if volley_lyon_all:
    dates_found = find_date_keys(volley_lyon_all[0])
    print("Clés avec dates/heures trouvées:")
    for key, value in dates_found:
        print(f"  {key}: {value}")

Clés avec dates/heures trouvées:
  infosRencontre.phase_parametrage.date_cloture_roster: None
  infosRencontre.phase_parametrage.updated_at: 2026-01-19 12:00:43
  infosRencontre.date_rencontre: 12/03/2026 14:00
  infosRencontre.score[0].updated_at: 2026-01-21 09:44:01
  score[0].updated_at: 2026-01-21 09:44:01
  chronometre.updated_at: 2026-01-21 09:44:01


In [57]:
# Filtrer les matchs de volley Lyon par la vraie date
target_date = "05/02/2026"

volley_5_fevrier = []
for match in volley_lyon_all:
    date_rencontre = match.get('infosRencontre', {}).get('date_rencontre', '')
    if target_date in str(date_rencontre):
        volley_5_fevrier.append(match)

print(f"Matchs Volley Lyon PH2 le {target_date}: {len(volley_5_fevrier)}")

# Si pas de matchs le 5 février, afficher les prochaines dates
if not volley_5_fevrier:
    dates_uniques = {}
    for match in volley_lyon_all:
        date_rencontre = match.get('infosRencontre', {}).get('date_rencontre', 'N/A')
        if date_rencontre not in dates_uniques:
            dates_uniques[date_rencontre] = 0
        dates_uniques[date_rencontre] += 1
    
    print("\nDates disponibles (triées):")
    for date, count in sorted(dates_uniques.items()):
        print(f"  {date}: {count} matchs")

Matchs Volley Lyon PH2 le 05/02/2026: 31


In [58]:
# Afficher les 31 matchs de volley du 5 février
print(f"🏐 Matchs de Volley Lyon PH2 le 05/02/2026 ({len(volley_5_fevrier)} matchs):\n")

for i, match in enumerate(volley_5_fevrier, 1):
    info = match.get('infosRencontre', {})
    receveur = match.get('receveur', {}).get('libelle', 'N/A')
    visiteur = match.get('visiteur', {}).get('libelle', 'N/A')
    date_rencontre = info.get('date_rencontre', 'N/A')
    lieu = info.get('lieu', {}).get('libelle', 'N/A') if isinstance(info.get('lieu'), dict) else info.get('lieu', 'N/A')
    comp = info.get('competition_libelle', '')
    match_id = match.get('id')
    
    # Extraire le nom court
    rec_short = receveur.split(' - ')[-1][:25] if ' - ' in receveur else receveur[:25]
    vis_short = visiteur.split(' - ')[-1][:25] if ' - ' in visiteur else visiteur[:25]
    
    genre = 'F' if 'VBF' in comp else 'M' if 'VBM' in comp else '?'
    
    print(f"{i:2}. [{genre}] {rec_short:25} vs {vis_short:25} | {date_rencontre} | ID: {match_id}")

🏐 Matchs de Volley Lyon PH2 le 05/02/2026 (31 matchs):

 1. [F] F                         vs AS INSA LYON VOLLEY-BALL  | 05/02/2026 14:00 | ID: 20185
 2. [F] LYON 1 (11)               vs LYON 1 (1)                | 05/02/2026 16:00 | ID: 20188
 3. [F] Lyon1Santé(1)             vs UDL- UTE LYON 2 VB F (1)  | 05/02/2026 20:00 | ID: 20203
 4. [F] EMSLB 1                   vs LYON3 (1) Volley Féminin  | 05/02/2026 20:00 | ID: 20206
 5. [F] 069069016 Centrale Lyon V vs AS INSA LYON VOLLEY-BALL  | 05/02/2026 16:00 | ID: 20211
 6. [F] LYON 1 (4)                vs Lyon1Santé(2)             | 05/02/2026 20:00 | ID: 20217
 7. [F] CPE                       vs LYON 1(6)                 | 05/02/2026 20:00 | ID: 20242
 8. [F] UDL- UTE LYON 2-IEP VB F  vs CATHO (1)                 | 05/02/2026 20:00 | ID: 20256
 9. [F] 069069016 CENTRALE LYON V vs LYON 1(7)                 | 05/02/2026 14:00 | ID: 20259
10. [F] Equipe féminine de volley vs LYON 1 (13)               | 05/02/2026 16:00 | ID: 20265
11. 

## Récupération des détails d'une feuille de match

Maintenant explorons l'endpoint pour récupérer les détails d'un match spécifique (joueurs, staff, arbitres, validations).

In [59]:
# Utiliser l'endpoint pour obtenir les détails d'un match
# Routes disponibles
print("Routes disponibles pour les détails:")
for name, url in routes.items():
    if 'rencontre' in url.lower() and 'rencontres' not in url.lower():
        print(f"  {name}: {url}")

Routes disponibles pour les détails:
  rencontre: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id
  participants_rencontres: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/participants
  dupliquer_joueurs: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/dupliquer/joueurs
  dupliquer_staffs: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/dupliquer/staffs
  update_rencontre: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id
  update_rencontre_detail: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/details
  recherche_personne: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/recherche-personne
  ajout_participant: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/participant/ajout
  supression_participant: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/participant/suppression
  update_participant: https:

In [60]:
# Tester l'endpoint rencontre avec un ID de match
test_match_id = volley_5_fevrier[0]['id']
rencontre_base_url = routes.get('rencontre')
print(f"URL de base: {rencontre_base_url}")

# Construire l'URL complète
rencontre_url = f"{BASE_URL}/feuille-de-match/rencontre/{test_match_id}"
print(f"URL complète: {rencontre_url}")

resp = session.get(rencontre_url, headers={
    'Accept': 'application/json',
    'X-Requested-With': 'XMLHttpRequest'
})
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type')}")

if resp.status_code == 200:
    if 'json' in resp.headers.get('Content-Type', ''):
        data = resp.json()
        print(f"\nClés principales:")
        for key in list(data.keys())[:10]:
            val = str(data[key])[:150]
            print(f"  {key}: {val}")

URL de base: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id
URL complète: https://gestion.mysportu.com/feuille-de-match/rencontre/20185
Status: 200
Content-Type: application/json

Clés principales:
  rencontre: {'id': 20185, 'validations': [{'sportif_rencontre_id': 20185, 'equipe_id': None, 'joueurs_valide': False, 'staffs_valide': False, 'officiels_valide': 
  boutons: [{'code': 'COUP_ENVOI', 'libelle': "Coup d'envoi", 'libelle_2': 'Saisie live', 'libelle_3': 'Déroulé de la rencontre', 'icone': 'icon-stack-text', 'ac
  types_penalites: []


In [61]:
# Explorer en détail les données d'une rencontre
data = resp.json()
rencontre = data.get('rencontre', {})

print("=== DÉTAILS DE LA RENCONTRE ===")
print(f"ID: {rencontre.get('id')}")

# Validations
print("\n=== VALIDATIONS ===")
validations = rencontre.get('validations', [])
for val in validations:
    print(f"  Équipe ID: {val.get('equipe_id')}")
    print(f"    - Joueurs validés: {val.get('joueurs_valide')}")
    print(f"    - Staffs validés: {val.get('staffs_valide')}")
    print(f"    - Officiels validés: {val.get('officiels_valide')}")

# Équipes avec joueurs
print("\n=== ÉQUIPES ===")
equipes = rencontre.get('equipes', [])
for equipe in equipes:
    print(f"\n  Équipe: {equipe.get('libelle', 'N/A')}")
    print(f"    Type: {'Receveur' if equipe.get('est_receveur') else 'Visiteur'}")
    
    joueurs = equipe.get('joueurs', [])
    print(f"    Joueurs inscrits: {len(joueurs)}")
    
    # Compter les joueurs sélectionnés
    joueurs_selectionnes = [j for j in joueurs if j.get('selectionne')]
    print(f"    Joueurs sélectionnés: {len(joueurs_selectionnes)}")
    
    # Afficher quelques joueurs
    for j in joueurs_selectionnes[:5]:
        nom = f"{j.get('prenom', '')} {j.get('nom', '')}"
        print(f"      - {nom}")

=== DÉTAILS DE LA RENCONTRE ===
ID: 20185

=== VALIDATIONS ===
  Équipe ID: None
    - Joueurs validés: False
    - Staffs validés: False
    - Officiels validés: True
  Équipe ID: 1075
    - Joueurs validés: False
    - Staffs validés: True
    - Officiels validés: False

=== ÉQUIPES ===


In [62]:
# Voir toutes les clés de la rencontre
print("Clés de la rencontre:")
for key in rencontre.keys():
    val = rencontre[key]
    if isinstance(val, list):
        print(f"  {key}: list[{len(val)}]")
    elif isinstance(val, dict):
        print(f"  {key}: dict with keys {list(val.keys())[:5]}")
    else:
        print(f"  {key}: {type(val).__name__} = {str(val)[:100]}")

Clés de la rencontre:
  id: int = 20185
  validations: list[2]
  rosters: dict with keys ['joueurs', 'staffs']
  receveur: dict with keys ['id', 'participant_id', 'libelle', 'libelle_court', 'logo']
  visiteur: dict with keys ['id', 'participant_id', 'libelle', 'libelle_court', 'logo']
  infosRencontre: dict with keys ['id', 'competition_libelle', 'competition_type', 'phase_libelle', 'phase_parametrage']
  regles: dict with keys ['nombre_joueurs_min', 'joueurs_minimum_options', 'nombre_joueurs_max', 'nombre_joueurs_transfert_max', 'nombre_joueurs_non_forme_fr']
  lieux_pratiques: list[0]
  officiels: list[1]
  lignes_equipes: list[2]
  medecin: list[0]
  chronometres: list[2]
  forfait: NoneType = None
  signatures: list[0]
  reserves: list[0]
  user: dict with keys ['id', 'est_federation', 'role', 'est_officiel', 'gestion_equipes_ids']


In [63]:
# Explorer les rosters (joueurs et staffs)
rosters = rencontre.get('rosters', {})
print("=== ROSTERS ===")
print(f"Clés: {rosters.keys()}")

# Joueurs
joueurs = rosters.get('joueurs', [])
print(f"\nJoueurs inscrits: {len(joueurs)}")

# Grouper par équipe
joueurs_par_equipe = {}
for j in joueurs:
    equipe_id = j.get('equipe_id')
    if equipe_id not in joueurs_par_equipe:
        joueurs_par_equipe[equipe_id] = []
    joueurs_par_equipe[equipe_id].append(j)

for equipe_id, jlist in joueurs_par_equipe.items():
    print(f"\n  Équipe ID {equipe_id}: {len(jlist)} joueurs")
    for j in jlist[:3]:
        nom = f"{j.get('prenom', '')} {j.get('nom', '')}"
        selectionne = "✅" if j.get('selectionne') else "❌"
        print(f"    {selectionne} {nom}")

# Staffs
staffs = rosters.get('staffs', [])
print(f"\n\nStaffs inscrits: {len(staffs)}")
staffs_par_equipe = {}
for s in staffs:
    equipe_id = s.get('equipe_id')
    if equipe_id not in staffs_par_equipe:
        staffs_par_equipe[equipe_id] = []
    staffs_par_equipe[equipe_id].append(s)

for equipe_id, slist in staffs_par_equipe.items():
    print(f"\n  Équipe ID {equipe_id}: {len(slist)} staffs")
    for s in slist[:3]:
        nom = f"{s.get('prenom', '')} {s.get('nom', '')}"
        fonction = s.get('fonction', 'N/A')
        selectionne = "✅" if s.get('selectionne') else "❌"
        print(f"    {selectionne} {nom} ({fonction})")

=== ROSTERS ===
Clés: dict_keys(['joueurs', 'staffs'])

Joueurs inscrits: 2

  Équipe ID 3794: 1 joueurs
    ❌  

  Équipe ID 1075: 1 joueurs
    ❌  


Staffs inscrits: 2

  Équipe ID 3794: 1 staffs
    ❌   (N/A)

  Équipe ID 1075: 1 staffs
    ❌   (N/A)


In [64]:
# Examiner la structure détaillée d'un joueur
if joueurs:
    print("Structure d'un joueur:")
    print(json.dumps(joueurs[0], indent=2, ensure_ascii=False, default=str))

Structure d'un joueur:
{
  "equipe_id": 3794,
  "licencies": []
}


In [65]:
# Examiner les officiels (arbitres)
officiels = rencontre.get('officiels', [])
print(f"Officiels: {len(officiels)}")

if officiels:
    print("\nStructure d'un officiel:")
    print(json.dumps(officiels[0], indent=2, ensure_ascii=False, default=str))

Officiels: 1

Structure d'un officiel:
{
  "id": 1014025,
  "nom": "SOUAK",
  "nom_complet": "Mme SOUAK Lydia",
  "prenom": "Lydia",
  "prenom_usage": null,
  "ddn": "29/07/2003",
  "code_adherent": "1014024",
  "attributs": [
    {
      "id": 1
    }
  ],
  "photo_url": "https://gestion.mysportu.com/storage/photos_personnes/miniature/1014025_514258_photo_id.png",
  "est_present": null,
  "est_suspendu": false,
  "licences": [
    {
      "libelle": "ARBITRE",
      "type_id": 83,
      "type_code": "A"
    }
  ],
  "formations": [],
  "est_notifie": true
}


In [ ]:
def analyser_feuille_match(session, match_id, base_url):
    """
    Analyse l'état de préparation d'une feuille de match.
    Utilise l'endpoint /participants pour obtenir le vrai nombre de joueurs.
    
    Returns:
        dict avec les informations sur l'état de préparation
    """
    # Récupérer les détails du match
    url = f"{base_url}/feuille-de-match/rencontre/{match_id}"
    resp = session.get(url, headers={
        'Accept': 'application/json',
        'X-Requested-With': 'XMLHttpRequest'
    })
    
    if resp.status_code != 200:
        return {'error': f"Status {resp.status_code}", 'id': match_id}
    
    data = resp.json()
    rencontre = data.get('rencontre', {})
    
    # Récupérer les participants (endpoint séparé)
    participants_url = f"{base_url}/feuille-de-match/rencontre/{match_id}/participants"
    resp_participants = session.get(participants_url, headers={
        'Accept': 'application/json',
        'X-Requested-With': 'XMLHttpRequest'
    })
    
    if resp_participants.status_code == 200:
        participants_data = resp_participants.json()
        joueurs_all = participants_data.get('joueurs', [])
        staffs_all = participants_data.get('staffs', [])
    else:
        joueurs_all = []
        staffs_all = []
    
    # Informations de base
    result = {
        'id': match_id,
        'receveur': rencontre.get('receveur', {}).get('libelle', 'N/A'),
        'visiteur': rencontre.get('visiteur', {}).get('libelle', 'N/A'),
        'equipes': {}
    }
    
    # Règles
    regles = rencontre.get('regles', {})
    min_joueurs = int(regles.get('nombre_joueurs_min', 6) or 6)
    result['min_joueurs'] = min_joueurs
    
    # Validations par équipe
    validations = {v.get('equipe_id'): v for v in rencontre.get('validations', [])}
    
    # Regrouper joueurs/staffs par équipe
    from collections import defaultdict
    joueurs_par_equipe = defaultdict(list)
    staffs_par_equipe = defaultdict(list)
    
    for j in joueurs_all:
        joueurs_par_equipe[j.get('equipe_id')].append(j)
    for s in staffs_all:
        staffs_par_equipe[s.get('equipe_id')].append(s)
    
    # Analyser chaque équipe
    for equipe_type in ['receveur', 'visiteur']:
        equipe_info = rencontre.get(equipe_type, {})
        equipe_id = equipe_info.get('id')
        equipe_nom = equipe_info.get('libelle_court', equipe_info.get('libelle', 'N/A'))
        
        # Validation de l'équipe
        val = validations.get(equipe_id, {})
        
        # Joueurs de cette équipe (depuis /participants)
        joueurs_equipe = joueurs_par_equipe.get(equipe_id, [])
        
        # Staffs de cette équipe (depuis /participants)
        staffs_equipe = staffs_par_equipe.get(equipe_id, [])
        
        result['equipes'][equipe_type] = {
            'id': equipe_id,
            'nom': equipe_nom,
            'joueurs_inscrits': len(joueurs_equipe),
            'joueurs_valide': val.get('joueurs_valide', False),
            'staffs_inscrits': len(staffs_equipe),
            'staffs_valide': val.get('staffs_valide', False),
        }
    
    # Officiels (arbitres)
    officiels = rencontre.get('officiels', [])
    officiels_presents = [o for o in officiels if o.get('est_present')]
    result['officiels'] = {
        'total': len(officiels),
        'presents': len(officiels_presents),
        'noms': [o.get('nom_complet', o.get('nom', 'N/A')) for o in officiels]
    }
    
    # Validation globale des officiels
    val_globale = validations.get(None, {})
    result['officiels_valide'] = val_globale.get('officiels_valide', False)
    
    # Statut global
    equipes_pretes = all(
        eq['joueurs_valide'] and eq['joueurs_inscrits'] >= min_joueurs
        for eq in result['equipes'].values()
    )
    arbitre_pret = result['officiels']['total'] > 0
    
    result['pret_a_jouer'] = equipes_pretes and arbitre_pret
    
    return result

# Tester sur un match
test_result = analyser_feuille_match(session, volley_5_fevrier[0]['id'], BASE_URL)
print(json.dumps(test_result, indent=2, ensure_ascii=False))

{
  "id": 20185,
  "receveur": "069069001 - ENTPE (1) - F",
  "visiteur": "069069025 - AS INSA LYON VOLLEY-BALL FEMININ 1",
  "equipes": {
    "receveur": {
      "id": 3794,
      "nom": "ENTPE (1)",
      "joueurs_total": 0,
      "joueurs_selectionnes": 0,
      "joueurs_valide": false,
      "staffs_total": 0,
      "staffs_selectionnes": 0,
      "staffs_valide": false
    },
    "visiteur": {
      "id": 1075,
      "nom": "INSA (1)",
      "joueurs_total": 0,
      "joueurs_selectionnes": 0,
      "joueurs_valide": false,
      "staffs_total": 0,
      "staffs_selectionnes": 0,
      "staffs_valide": true
    }
  },
  "min_joueurs": 6,
  "officiels": {
    "total": 1,
    "presents": 0,
    "noms": [
      "Mme SOUAK Lydia"
    ]
  },
  "officiels_valide": true,
  "pret_a_jouer": false
}


In [69]:
# Analyser tous les matchs du 5 février
from tqdm import tqdm
import time

print(f"Analyse de {len(volley_5_fevrier)} matchs de Volley Lyon PH2 du 05/02/2026...\n")

resultats = []
for match in tqdm(volley_5_fevrier):
    match_id = match['id']
    result = analyser_feuille_match(session, match_id, BASE_URL)
    result['date'] = match.get('infosRencontre', {}).get('date_rencontre', 'N/A')
    result['competition'] = match.get('infosRencontre', {}).get('competition_libelle', 'N/A')
    resultats.append(result)
    time.sleep(0.1)  # Pause pour ne pas surcharger l'API

print(f"\nAnalyse terminée: {len(resultats)} matchs analysés")

Analyse de 31 matchs de Volley Lyon PH2 du 05/02/2026...



100%|██████████| 31/31 [00:14<00:00,  2.12it/s]


Analyse terminée: 31 matchs analysés


In [71]:
# Créer un DataFrame récapitulatif
recap_data = []
for r in resultats:
    if 'error' in r:
        recap_data.append({
            'ID': r['id'],
            'Receveur': 'ERREUR',
            'Visiteur': r['error'],
            'Prêt': '❌'
        })
        continue
    
    rec = r['equipes'].get('receveur', {})
    vis = r['equipes'].get('visiteur', {})
    
    rec_status = f"J:{rec.get('joueurs_selectionnes', 0)}/{r.get('min_joueurs', 6)} "
    rec_status += "✓" if rec.get('joueurs_valide') else "✗"
    rec_status += f" S:{rec.get('staffs_selectionnes', 0)} "
    rec_status += "✓" if rec.get('staffs_valide') else "✗"
    
    vis_status = f"J:{vis.get('joueurs_selectionnes', 0)}/{r.get('min_joueurs', 6)} "
    vis_status += "✓" if vis.get('joueurs_valide') else "✗"
    vis_status += f" S:{vis.get('staffs_selectionnes', 0)} "
    vis_status += "✓" if vis.get('staffs_valide') else "✗"
    
    arb_status = f"{r['officiels']['total']} arbitre(s)"
    
    recap_data.append({
        'ID': r['id'],
        'Receveur': rec.get('nom', 'N/A'),
        'Rec Status': rec_status,
        'Visiteur': vis.get('nom', 'N/A'),
        'Vis Status': vis_status,
        'Arbitre': arb_status,
        'Prêt': '✅' if r['pret_a_jouer'] else '❌'
    })

df_recap = pd.DataFrame(recap_data)
print(f"=== RÉCAPITULATIF DES {len(df_recap)} MATCHS DE VOLLEY LYON PH2 - 05/02/2026 ===\n")
print(f"Matchs prêts: {len(df_recap[df_recap['Prêt'] == '✅'])} / {len(df_recap)}")
print(f"Matchs non prêts: {len(df_recap[df_recap['Prêt'] == '❌'])} / {len(df_recap)}")
print()
df_recap

=== RÉCAPITULATIF DES 31 MATCHS DE VOLLEY LYON PH2 - 05/02/2026 ===

Matchs prêts: 0 / 31
Matchs non prêts: 31 / 31



,ID,Receveur,Rec Status,Visiteur,Vis Status,Arbitre,Prêt
0,20185,ENTPE (1),J:0/6 ✗ S:0 ✗,INSA (1),J:0/6 ✗ S:0 ✓,1 arbitre(s),❌
1,20188,LYON 1 (11),J:0/6 ✗ S:0 ✗,LYON 1 (1),J:0/6 ✗ S:0 ✗,1 arbitre(s),❌
2,20203,SANTE (1),J:0/6 ✗ S:0 ✗,LYON 2 (1),J:0/6 ✗ S:0 ✗,0 arbitre(s),❌
3,20206,ESA (1),J:0/6 ✗ S:0 ✗,LYON 3 (1),J:0/6 ✗ S:0 ✓,1 arbitre(s),❌
4,20211,ECL (1),J:0/6 ✗ S:0 ✗,INSA (3),J:0/6 ✗ S:0 ✓,1 arbitre(s),❌
5,20217,LYON 1 (4),J:0/6 ✓ S:0 ✓,SANTE (2),J:0/6 ✗ S:0 ✗,1 arbitre(s),❌
6,20242,CPE (1),J:0/6 ✗ S:0 ✗,LYON 1 (6),J:0/6 ✗ S:0 ✗,1 arbitre(s),❌
7,20256,LYON 2 (IEP) (4),J:0/6 ✗ S:0 ✗,CATHO (1),J:0/6 ✗ S:0 ✗,0 arbitre(s),❌
8,20259,ECL (2),J:0/6 ✗ S:0 ✗,LYON 1 (7),J:0/6 ✗ S:0 ✗,1 arbitre(s),❌
9,20265,ESSCA (1),J:0/6 ✗ S:0 ✗,LYON 1 (13),J:0/6 ✗ S:0 ✗,0 arbitre(s),❌


## Exploration approfondie - Recherche des joueurs

In [72]:
# Afficher toutes les routes disponibles
print("=== ROUTES API DISPONIBLES ===\n")
for key, value in routes.items():
    print(f"{key}: {value}")

=== ROUTES API DISPONIBLES ===

liste_rencontres: https://gestion.mysportu.com/feuille-de-match/rencontres
export_rencontres: https://gestion.mysportu.com/feuille-de-match/rencontres/export
liste_competitions: https://gestion.mysportu.com/feuille-de-match/ajax/competitions
liste_structures: https://gestion.mysportu.com/feuille-de-match/ajax/structures
liste_disciplines: https://gestion.mysportu.com/feuille-de-match/ajax/disciplines
rencontre: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id
participants_rencontres: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/participants
dupliquer_joueurs: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/dupliquer/joueurs
dupliquer_staffs: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id/dupliquer/staffs
update_rencontre: https://gestion.mysportu.com/feuille-de-match/rencontre/rencontre_id
update_rencontre_detail: https://gestion.mysportu.com/feuille-de-match/rencontre

In [73]:
# Tester l'endpoint participants pour un match
test_id = volley_5_fevrier[0]['id']
print(f"Test avec le match ID: {test_id}\n")

# Essayer l'endpoint participants
participants_url = f"{BASE_URL}/feuille-de-match/rencontre/{test_id}/participants"
resp = session.get(participants_url, headers={
    'Accept': 'application/json',
    'X-Requested-With': 'XMLHttpRequest'
})
print(f"GET {participants_url}")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type', 'N/A')}")

if resp.status_code == 200:
    try:
        data = resp.json()
        print(f"\nClés: {list(data.keys())}")
        print(f"\nDonnées brutes (aperçu):")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])
    except:
        print(f"\nRéponse texte: {resp.text[:500]}")

Test avec le match ID: 20185

GET https://gestion.mysportu.com/feuille-de-match/rencontre/20185/participants
Status: 200
Content-Type: application/json

Clés: ['joueurs', 'staffs']

Données brutes (aperçu):
{
  "joueurs": [
    {
      "participant_id": 27057,
      "id": 839254,
      "nom_complet": "Mme ADAM Manon",
      "nom": "ADAM",
      "prenom": "Manon",
      "pseudo": "",
      "prenom_usage": null,
      "ddn": "11/06/2005",
      "code_adherent": "0839253",
      "attributs": [],
      "photo_url": "https://gestion.mysportu.com/storage/photos_personnes/miniature/839254_427049_photo_id.png",
      "titulaire": false,
      "equipe_id": 1075,
      "est_present": null,
      "ligne_id": 3499,
      "a_joue": null,
      "numero": "7",
      "est_jfl": false,
      "est_suspendu": false,
      "est_suspendu_medicale": false,
      "etranger": false,
      "mutee": false,
      "licences": [
        {
          "libelle": "SPORTIVE",
          "type_id": 81,
          "type_co

In [74]:
# Analyser la structure des participants
participants_data = resp.json()
joueurs = participants_data.get('joueurs', [])
staffs = participants_data.get('staffs', [])

print(f"=== PARTICIPANTS DU MATCH {test_id} ===\n")
print(f"Nombre total de joueurs inscrits: {len(joueurs)}")
print(f"Nombre total de staffs inscrits: {len(staffs)}")

# Regrouper par équipe
from collections import defaultdict
joueurs_par_equipe = defaultdict(list)
staffs_par_equipe = defaultdict(list)

for j in joueurs:
    joueurs_par_equipe[j.get('equipe_id')].append(j)
for s in staffs:
    staffs_par_equipe[s.get('equipe_id')].append(s)

print(f"\n=== PAR ÉQUIPE ===")
for equipe_id in joueurs_par_equipe:
    joueurs_eq = joueurs_par_equipe[equipe_id]
    staffs_eq = staffs_par_equipe.get(equipe_id, [])
    print(f"\nÉquipe ID {equipe_id}:")
    print(f"  - Joueurs: {len(joueurs_eq)}")
    print(f"  - Staffs: {len(staffs_eq)}")
    
    # Afficher quelques exemples
    for j in joueurs_eq[:3]:
        print(f"    -> {j.get('nom_complet')} (n°{j.get('numero', 'N/A')}) - titulaire: {j.get('titulaire')}")

=== PARTICIPANTS DU MATCH 20185 ===

Nombre total de joueurs inscrits: 18
Nombre total de staffs inscrits: 1

=== PAR ÉQUIPE ===

Équipe ID 1075:
  - Joueurs: 10
  - Staffs: 1
    -> Mme ADAM Manon (n°7) - titulaire: False
    -> Mme CADEOT Naïs (n°11) - titulaire: False
    -> Mme CORREAS GIL Ada Wei (n°5) - titulaire: False

Équipe ID 3794:
  - Joueurs: 8
  - Staffs: 0
    -> Mme LECUYER Ombeline (n°None) - titulaire: False
    -> Mme AUJOULAT Clemence (n°None) - titulaire: False
    -> Mme DREYFUS Julia (n°None) - titulaire: False


In [75]:
# Comparer avec les données de rencontre pour comprendre la structure complète
rencontre_url = f"{BASE_URL}/feuille-de-match/rencontre/{test_id}"
resp_rencontre = session.get(rencontre_url, headers={
    'Accept': 'application/json',
    'X-Requested-With': 'XMLHttpRequest'
})
rencontre_data = resp_rencontre.json()
rencontre = rencontre_data.get('rencontre', {})

print("=== COMPARAISON DES DONNÉES ===\n")

# Infos sur les équipes
receveur = rencontre.get('receveur', {})
visiteur = rencontre.get('visiteur', {})
print(f"Receveur: {receveur.get('libelle')} (ID: {receveur.get('id')})")
print(f"Visiteur: {visiteur.get('libelle')} (ID: {visiteur.get('id')})")

# Validations
validations = rencontre.get('validations', [])
print(f"\n=== VALIDATIONS ===")
for v in validations:
    equipe_id = v.get('equipe_id')
    equipe_nom = "Receveur" if equipe_id == receveur.get('id') else "Visiteur" if equipe_id == visiteur.get('id') else f"ID {equipe_id}"
    print(f"\n{equipe_nom}:")
    print(f"  joueurs_valide: {v.get('joueurs_valide')}")
    print(f"  staffs_valide: {v.get('staffs_valide')}")
    print(f"  nb_joueurs: {v.get('nb_joueurs')}")
    print(f"  nb_staffs: {v.get('nb_staffs')}")

# Règles
regles = rencontre.get('regles', {})
print(f"\n=== RÈGLES ===")
print(f"nombre_joueurs_min: {regles.get('nombre_joueurs_min')}")
print(f"nombre_joueurs_max: {regles.get('nombre_joueurs_max')}")
print(f"nombre_titulaires: {regles.get('nombre_titulaires')}")
print(f"nombre_staffs_min: {regles.get('nombre_staffs_min')}")

=== COMPARAISON DES DONNÉES ===

Receveur: 069069001 - ENTPE (1) - F (ID: 3794)
Visiteur: 069069025 - AS INSA LYON VOLLEY-BALL FEMININ 1 (ID: 1075)

=== VALIDATIONS ===

ID None:
  joueurs_valide: False
  staffs_valide: False
  nb_joueurs: None
  nb_staffs: None

Visiteur:
  joueurs_valide: False
  staffs_valide: True
  nb_joueurs: None
  nb_staffs: None

=== RÈGLES ===
nombre_joueurs_min: 6
nombre_joueurs_max: 14
nombre_titulaires: None
nombre_staffs_min: None


In [76]:
# Lister les compétitions disponibles
competitions_url = f"{BASE_URL}/feuille-de-match/ajax/competitions"
resp_comp = session.get(competitions_url, headers={
    'Accept': 'application/json',
    'X-Requested-With': 'XMLHttpRequest'
})

print(f"Status: {resp_comp.status_code}\n")
if resp_comp.status_code == 200:
    competitions_list = resp_comp.json()
    print(f"Nombre de compétitions: {len(competitions_list)}\n")
    
    # Grouper par type de sport
    from collections import defaultdict
    by_sport = defaultdict(list)
    for c in competitions_list:
        libelle = c.get('libelle', 'N/A')
        # Essayer de déduire le sport du libellé
        sport = 'Autre'
        if 'VB' in libelle or 'VOLLEY' in libelle.upper():
            sport = 'Volleyball'
        elif 'HB' in libelle or 'HAND' in libelle.upper():
            sport = 'Handball'
        elif 'BB' in libelle or 'BASKET' in libelle.upper():
            sport = 'Basketball'
        elif 'FB' in libelle or 'FOOT' in libelle.upper():
            sport = 'Football'
        by_sport[sport].append(c)
    
    for sport, comps in sorted(by_sport.items()):
        print(f"\n=== {sport.upper()} ({len(comps)} compétitions) ===")
        for c in comps[:10]:  # Limiter l'affichage
            print(f"  ID {c.get('id')}: {c.get('libelle')}")

Status: 200

Nombre de compétitions: 750


=== AUTRE (220 compétitions) ===
  ID 752: Championnat de France Universitaire de rugby 10 Masculin 2026
  ID 771: CENTRE EST Championnat Inter-Ligues Universitaire de rugby 10 Masculin
  ID 184: Championnat IDF Universitaire de rugby 10 Masculin
  ID 678: Championnat Inter-Ligues Universitaire de Rugby 10 Masculin Nord-Est (Grand-Est, Bourgogne-Franche-Comté, Hauts-de-France)
  ID 686: Championnat Inter-Ligues Sud-Ouest Universitaire de rugby 10 Masculin
  ID 452: 1ère Phase Académique CFE RG XV JG - Poitiers/La Rochelle
  ID 73: ANGERS District FUTSAL Masculin
  ID 209: ANGERS District RUGBY à 15 Masculin
  ID 207: ANGERS District RUGBY à 7 Féminin
  ID 208: ANGERS District RUGBY à 7 Masculin

=== BASKETBALL (127 compétitions) ===
  ID 461: 1ère Phase Académique CFE BB JF - Poitiers/La Rochelle
  ID 454: 1ère Phase Académique CFE BB JG - Poitiers/La Rochelle
  ID 476: ANGERS District Basket-ball 5x5
  ID 962: ANGERS Trophée de la Ville BASKE

## Exploration des statuts de match (reporté, forfait, etc.)

In [77]:
# Analyser la structure d'un match pour trouver les indicateurs de statut
# Prendre un match de la liste
sample_match = volley_5_fevrier[0]
print("=== STRUCTURE D'UN MATCH (niveau liste) ===\n")
print(f"Clés disponibles: {list(sample_match.keys())}")
print()

# Afficher les champs intéressants
for key in sample_match.keys():
    value = sample_match[key]
    if isinstance(value, dict):
        print(f"{key}: (dict avec {len(value)} clés: {list(value.keys())[:5]}...)")
    elif isinstance(value, list):
        print(f"{key}: (list avec {len(value)} éléments)")
    else:
        print(f"{key}: {value}")

=== STRUCTURE D'UN MATCH (niveau liste) ===

Clés disponibles: ['id', 'receveur', 'visiteur', 'infosRencontre', 'score', 'sets', 'chronometre', 'etat', 'forfait', 'tour', 'poule', 'clos']

id: 20185
receveur: (dict avec 11 clés: ['id', 'participant_id', 'libelle', 'libelle_court', 'logo']...)
visiteur: (dict avec 11 clés: ['id', 'participant_id', 'libelle', 'libelle_court', 'logo']...)
infosRencontre: (dict avec 26 clés: ['id', 'competition_libelle', 'competition_type', 'phase_libelle', 'phase_parametrage']...)
score: (list avec 2 éléments)
sets: (list avec 0 éléments)
chronometre: (dict avec 10 clés: ['id', 'sportif_rencontre_id', 'reference_debut', 'milisecondes_total', 'en_pause']...)
etat: None
forfait: None
tour: 1
poule: (dict avec 5 clés: ['id', 'phase_id', 'position', 'libelle', 'deleted_at']...)
clos: False


In [79]:
# Analyser les différentes valeurs de 'etat', 'forfait', 'clos' sur tous les matchs
from collections import Counter

etats = Counter()
forfaits_none = 0
forfaits_dict = 0
clos_values = Counter()

for m in all_rencontres:
    etats[m.get('etat')] += 1
    forfait = m.get('forfait')
    if forfait is None:
        forfaits_none += 1
    else:
        forfaits_dict += 1
    clos_values[m.get('clos')] += 1

print("=== VALEURS 'etat' ===")
for val, count in etats.most_common():
    print(f"  {val!r}: {count} matchs")

print("\n=== VALEURS 'forfait' ===")
print(f"  None: {forfaits_none} matchs")
print(f"  Dict (forfait déclaré): {forfaits_dict} matchs")

print("\n=== VALEURS 'clos' ===")
for val, count in clos_values.most_common():
    print(f"  {val!r}: {count} matchs")

# Chercher un match avec forfait pour voir la structure
for m in all_rencontres:
    if m.get('forfait') is not None:
        print("\n=== EXEMPLE DE FORFAIT ===")
        print(json.dumps(m.get('forfait'), indent=2, ensure_ascii=False))
        break

=== VALEURS 'etat' ===
  None: 1361 matchs
  'T': 570 matchs
  'R': 3 matchs
  'N': 1 matchs

=== VALEURS 'forfait' ===
  None: 1920 matchs
  Dict (forfait déclaré): 15 matchs

=== VALEURS 'clos' ===
  False: 1425 matchs
  True: 510 matchs

=== EXEMPLE DE FORFAIT ===
{
  "id": 30,
  "sportif_rencontre_id": 6386,
  "user_id": 356,
  "equipe_id": 2375,
  "participant_id": 1399,
  "date_application": "2025-11-27 00:00:00",
  "commentaire": "Equipe pas au complet",
  "type": "INI",
  "created_at": "2025-11-26 14:52:10",
  "updated_at": "2025-11-26 14:52:10"
}


In [80]:
# Regarder les infosRencontre pour d'autres indicateurs
sample = all_rencontres[0]
print("=== CLÉS DANS infosRencontre ===")
infos = sample.get('infosRencontre', {})
for key, value in infos.items():
    if isinstance(value, (dict, list)):
        print(f"{key}: ({type(value).__name__})")
    else:
        print(f"{key}: {value!r}")

# Chercher un match reporté pour voir sa structure
print("\n=== MATCHS REPORTÉS (etat='R') ===")
for m in all_rencontres:
    if m.get('etat') == 'R':
        rec = m.get('receveur', {}).get('libelle_court', 'N/A')
        vis = m.get('visiteur', {}).get('libelle_court', 'N/A')
        date = m.get('infosRencontre', {}).get('date_rencontre', 'N/A')
        comp = m.get('infosRencontre', {}).get('competition_libelle', 'N/A')
        print(f"  ID {m['id']}: {rec} vs {vis} - {date} ({comp})")

=== CLÉS DANS infosRencontre ===
id: 4070
competition_libelle: 'Championnat de France Universitaire de football Masculin N1'
competition_type: 'ELicence\\Sportif\\Models\\Competitions\\SportifCompetition'
phase_libelle: 'Phase de poules'
phase_parametrage: (dict)
phase_type: 'P'
discipline: (dict)
categories_ages: (list)
lieu_pratique: (dict)
date_rencontre: '13/11/2025 14:30'
ouverture_vestiaire: None
couleurs: (list)
score: (list)
sets: (list)
resultats_serie: (list)
score_cumule: (list)
nombre_rencontres_serie: 1
numero_serie: 1
prolongations: (list)
nombre_spectateurs: None
etat: 'T'
clos: 1
url_live: None
start_at: None
end_at: None
numero_match: None

=== MATCHS REPORTÉS (etat='R') ===
  ID 20265: ESSCA (1) vs LYON 1 (13) - 05/02/2026 16:00 (LYON VBF PH2 - Championnat Académique de Volley Féminin)
  ID 21399: LYON 1 (22) vs CATHO (1) - 05/02/2026 18:00 (LYON VBM PH2 - Championnat Académique Volley Masculin)
  ID 21434: LYON 1 (27) vs LYON 1 (14) - 29/01/2026 14:00 (LYON VBM PH2 -

In [81]:
# Chercher le match avec etat='N'
print("=== MATCH AVEC etat='N' ===")
for m in all_rencontres:
    if m.get('etat') == 'N':
        rec = m.get('receveur', {}).get('libelle_court', 'N/A')
        vis = m.get('visiteur', {}).get('libelle_court', 'N/A')
        date = m.get('infosRencontre', {}).get('date_rencontre', 'N/A')
        comp = m.get('infosRencontre', {}).get('competition_libelle', 'N/A')
        print(f"  ID {m['id']}: {rec} vs {vis} - {date} ({comp})")
        print(f"  forfait: {m.get('forfait')}")
        print(f"  clos: {m.get('clos')}")

# Lister les matchs avec forfait
print("\n=== MATCHS AVEC FORFAIT ===")
for m in all_rencontres:
    if m.get('forfait') is not None:
        rec = m.get('receveur', {}).get('libelle_court', 'N/A')
        vis = m.get('visiteur', {}).get('libelle_court', 'N/A')
        date = m.get('infosRencontre', {}).get('date_rencontre', 'N/A')
        forfait = m.get('forfait', {})
        forfait_type = forfait.get('type', 'N/A')
        forfait_equipe = forfait.get('equipe_id', 'N/A')
        print(f"  ID {m['id']}: {rec} vs {vis} - {date} (type: {forfait_type})")

=== MATCH AVEC etat='N' ===
  ID 9918: Grenoble EM vs 038038002 - UGA NB1 - 16/10/2025 14:00 (Acad Foot A 8 Féminin)
  forfait: None
  clos: False

=== MATCHS AVEC FORFAIT ===
  ID 6386: INP Grenoble vs 038073001 - USMB2 - 27/11/2025 14:30 (type: INI)
  ID 7216: GEM vs UGA Elisa - 23/10/2025 17:45 (type: INI)
  ID 7219: UGA Elisa vs Archi - 06/11/2025 19:00 (type: INI)
  ID 7713: UJM STAPS Ste vs LYON 3 (1) - 20/11/2025 13:30 (type: INI)
  ID 10475: 069042003 - STAPS 1 - Foufous vs 069042003 - STAPS 2 - Tigres - 10/11/2025 19:30 (type: INI)
  ID 10481: 069042003 - STAPS 1 - Foufous vs 069042005 - UJM - 24/11/2025 19:30 (type: INI)
  ID 10490: 069042003 - STAPS 3 - Olympique vs 069042006 - MINES - 04/12/2025 13:30 (type: INI)
  ID 12091: 038038006 - GEM F vs 038038002 - UGA 1 - 20/10/2025 20:30 (type: INI)
  ID 14642: 063063001 - Clermont BS Basket 5X5 vs GEM - 30/11/2025 12:00 (type: INI)
  ID 14643: GEM vs BASKET CLUB HEC FILLES 1 - 30/11/2025 14:00 (type: INI)
  ID 14661: 031031036 -

In [82]:
# Analyser les noms d'équipes pour extraire l'institution
# Format typique: "LYON 1 (4)", "ENTPE (1)", "ECL (2)", etc.
import re

# Regarder les noms des équipes dans volley_5_fevrier
print("=== NOMS D'ÉQUIPES (échantillon) ===")
noms = set()
for m in volley_5_fevrier[:20]:
    rec = m.get('receveur', {}).get('libelle_court', 'N/A')
    vis = m.get('visiteur', {}).get('libelle_court', 'N/A')
    noms.add(rec)
    noms.add(vis)

for nom in sorted(noms):
    print(f"  {nom}")

# Fonction pour extraire l'institution
def extraire_institution(nom_equipe):
    """Extrait l'institution du nom de l'équipe (ex: 'LYON 1 (4)' -> 'LYON 1')"""
    # Enlever le numéro d'équipe entre parenthèses
    match = re.match(r'^(.+?)\s*\(\d+\)$', nom_equipe)
    if match:
        return match.group(1).strip()
    return nom_equipe

print("\n=== TEST D'EXTRACTION ===")
test_noms = ['LYON 1 (4)', 'ENTPE (1)', 'ECL (2)', 'LYON 2 (IEP) (4)', 'INSA (3)', 'SANTE (1)']
for nom in test_noms:
    print(f"  '{nom}' -> '{extraire_institution(nom)}'")

=== NOMS D'ÉQUIPES (échantillon) ===
  CATHO (1)
  CPE (1)
  ECL (1)
  ECL (2)
  ECL (3)
  EML (2)
  ENTPE (1)
  ESA (1)
  ESA (2)
  ESME (1)
  ESME (2)
  ESSCA (1)
  INSA (1)
  INSA (3)
  INSA (4)
  LYON 1 (1)
  LYON 1 (10)
  LYON 1 (11)
  LYON 1 (13)
  LYON 1 (21)
  LYON 1 (26)
  LYON 1 (4)
  LYON 1 (5)
  LYON 1 (6)
  LYON 1 (7)
  LYON 1 (8)
  LYON 2 (1)
  LYON 2 (IEP) (4)
  LYON 3 (1)
  SANTE (1)
  SANTE (2)

=== TEST D'EXTRACTION ===
  'LYON 1 (4)' -> 'LYON 1'
  'ENTPE (1)' -> 'ENTPE'
  'ECL (2)' -> 'ECL'
  'LYON 2 (IEP) (4)' -> 'LYON 2 (IEP)'
  'INSA (3)' -> 'INSA'
  'SANTE (1)' -> 'SANTE'


In [6]:
# Récupérer la page de login pour obtenir le token CSRF
response = session.get(LOGIN_URL)
print(f"Status: {response.status_code}")
print(f"URL: {response.url}")

# Parser la page pour trouver le token CSRF
soup = BeautifulSoup(response.text, 'html.parser')

# Chercher le token CSRF
csrf_token = None
csrf_input = soup.find('input', {'name': '_token'})
if csrf_input:
    csrf_token = csrf_input.get('value')
    print(f"Token CSRF trouvé: {csrf_token[:20]}...")
else:
    print("Token CSRF non trouvé")
    # Afficher les inputs disponibles
    for inp in soup.find_all('input'):
        print(f"Input: {inp.get('name')} = {inp.get('value', '')[:30] if inp.get('value') else ''}")

Status: 404
URL: https://gestion.mysportu.com/login
Token CSRF non trouvé


In [ ]:
# Tentative de connexion
login_data = {
    '_token': csrf_token,
    'email': USERNAME,
    'password': PASSWORD,
    'remember': 'on'
}

response = session.post(LOGIN_URL, data=login_data, allow_redirects=True)
print(f"Status: {response.status_code}")
print(f"URL après login: {response.url}")
print(f"Cookies: {dict(session.cookies)}")

In [ ]:
# Vérifier si la connexion a réussi en accédant au dashboard
dashboard_url = f"{BASE_URL}/dashboard"
response = session.get(dashboard_url)
print(f"Status dashboard: {response.status_code}")
print(f"URL: {response.url}")

# Vérifier si on est bien connecté
if 'login' in response.url.lower():
    print("❌ Non connecté - redirigé vers login")
else:
    print("✅ Connexion réussie!")
    soup = BeautifulSoup(response.text, 'html.parser')
    title = soup.find('title')
    print(f"Titre de la page: {title.text if title else 'N/A'}")

## 2. Explorer les endpoints disponibles

In [ ]:
# Lister les liens disponibles sur le dashboard
soup = BeautifulSoup(response.text, 'html.parser')

# Trouver tous les liens
links = soup.find_all('a', href=True)
unique_links = set()
for link in links:
    href = link['href']
    if href.startswith('/') or 'mysportu' in href:
        unique_links.add(href)

print("Liens trouvés:")
for link in sorted(unique_links):
    print(f"  {link}")

In [ ]:
# Explorer les routes possibles pour les rencontres/matchs
possible_endpoints = [
    '/rencontres',
    '/matches',
    '/matchs',
    '/competitions',
    '/calendrier',
    '/calendar',
    '/journees',
    '/feuilles-de-match',
    '/feuille-match',
    '/api/rencontres',
    '/api/matches',
    '/api/competitions',
]

for endpoint in possible_endpoints:
    url = f"{BASE_URL}{endpoint}"
    try:
        resp = session.get(url, allow_redirects=False)
        print(f"{endpoint}: {resp.status_code}")
    except Exception as e:
        print(f"{endpoint}: Erreur - {e}")

## 3. Recherche de compétitions Volley 6x6

In [ ]:
# Explorer le contenu de la page dashboard pour mieux comprendre la structure
soup = BeautifulSoup(response.text, 'html.parser')

# Chercher des éléments liés aux matchs/rencontres
keywords = ['match', 'rencontre', 'volley', 'competition', 'journee', 'calendrier']
for keyword in keywords:
    elements = soup.find_all(string=lambda text: text and keyword.lower() in text.lower())
    if elements:
        print(f"\n=== {keyword.upper()} ===")
        for elem in elements[:5]:
            print(f"  {elem.strip()[:100]}")

In [ ]:
# Afficher une partie du HTML pour mieux comprendre la structure
print(response.text[:5000])

In [ ]:
# Chercher s'il y a des scripts JavaScript avec des données
scripts = soup.find_all('script')
for script in scripts:
    if script.string and ('match' in script.string.lower() or 'rencontre' in script.string.lower() or 'api' in script.string.lower()):
        print("Script trouvé avec données potentielles:")
        print(script.string[:2000] if len(script.string) > 2000 else script.string)
        print("---")

## 4. Test d'accès aux feuilles de match

In [ ]:
# Essayer différentes URLs pour accéder aux feuilles de match
test_urls = [
    '/gestion/feuille-match',
    '/gestion/rencontres',
    '/organisateur/rencontres',
    '/organisateur/feuilles-de-match',
    '/equipes',
    '/mes-equipes',
    '/my-teams',
]

for url_path in test_urls:
    url = f"{BASE_URL}{url_path}"
    try:
        resp = session.get(url)
        print(f"{url_path}: {resp.status_code} - URL finale: {resp.url}")
        if resp.status_code == 200 and 'login' not in resp.url:
            soup_temp = BeautifulSoup(resp.text, 'html.parser')
            title = soup_temp.find('title')
            print(f"  Titre: {title.text if title else 'N/A'}")
    except Exception as e:
        print(f"{url_path}: Erreur - {e}")

In [ ]:
# Explorer la navigation du site
# Rechercher des menus de navigation
soup = BeautifulSoup(response.text, 'html.parser')

# Chercher les éléments nav
navs = soup.find_all(['nav', 'aside', 'menu'])
print(f"Éléments de navigation trouvés: {len(navs)}")

for nav in navs:
    links_in_nav = nav.find_all('a', href=True)
    for link in links_in_nav:
        print(f"  {link.get('href')} - {link.get_text(strip=True)[:50]}")

In [ ]:
# Chercher dans les sidebars ou menus avec classes communes
sidebar_classes = ['sidebar', 'menu', 'nav', 'navigation']
for cls in sidebar_classes:
    elements = soup.find_all(class_=lambda x: x and cls in str(x).lower())
    if elements:
        print(f"\n=== Éléments avec classe '{cls}' ===")
        for elem in elements[:3]:
            links = elem.find_all('a', href=True)
            for link in links[:10]:
                print(f"  {link.get('href')} - {link.get_text(strip=True)[:40]}")

## 5. Recherche de l'API JSON

In [ ]:
# Tester des endpoints API potentiels
api_endpoints = [
    '/api/v1/matches',
    '/api/v1/rencontres',
    '/api/matches',
    '/api/rencontres',
    '/api/competitions',
    '/api/user',
    '/api/me',
    '/api/v1/user',
    '/api/v1/competitions',
]

for endpoint in api_endpoints:
    url = f"{BASE_URL}{endpoint}"
    try:
        resp = session.get(url, headers={'Accept': 'application/json'})
        print(f"{endpoint}: {resp.status_code}")
        if resp.status_code == 200:
            try:
                data = resp.json()
                print(f"  JSON reçu: {str(data)[:200]}")
            except:
                print(f"  Pas de JSON, HTML reçu")
    except Exception as e:
        print(f"{endpoint}: Erreur - {e}")

In [ ]:
# Chercher des appels XHR/AJAX dans le JavaScript
import re

soup = BeautifulSoup(response.text, 'html.parser')
scripts = soup.find_all('script')

api_patterns = [
    r'/api/[^"\s]+',
    r'fetch\(["\']([^"\']*)["\'\)]',
    r'axios\.[a-z]+\(["\']([^"\']*)["\'\)]',
    r'\$\.ajax\([^)]*url:\s*["\']([^"\']*)["\'\)]',
]

found_apis = set()
for script in scripts:
    if script.string:
        for pattern in api_patterns:
            matches = re.findall(pattern, script.string)
            found_apis.update(matches)

print("APIs trouvées dans les scripts:")
for api in sorted(found_apis):
    print(f"  {api}")

## 6. Exploration de la page principale

In [ ]:
# Afficher le HTML complet pour analyser la structure
print(response.text)